# 05. Analysis, charts and robustness (final notebook)
**Case study:** Partisanship & economic perception · **Layer:** presentation (marts → charts, tables, story numbers)

This is the last notebook in the pipeline. Notebooks 01–04 turned raw files into tested tables; this one **only reads
those tables** and turns them into everything the panel and the reader see:

| Section | Output | Answers | Reads |
|---|---|---|---|
| 2 | **Number sheet** | "Where does this number come from?" | all marts |
| 3 | **Lede chart**: UMich monthly party lines + Pew cross-check | RQ1, the flagship picture | `fct_sentiment_monthly`, `fct_party_ratings` |
| 4 | **Event study** + swing-timing intervals | RQ1: when does opinion flip? | `fct_event_study`, `fct_swing_timing_bootstrap` |
| 5 | **Fundamentals table** + "what moved" chart | RQ1: did the economy move? | `fct_fundamentals_at_transitions` |
| 6 | **CES annual series** + annual-drift yardstick + common-vs-partisan split | RQ1 (survey microdata) | `fct_party_ratings`, `fct_wave_contrasts` |
| 7 | **Midterm placebo** dot plot | RQ1 validity: does opinion follow Congress too? | `fct_midterm_placebo` |
| 8 | **Co-partisan premium** forest plot + year effects | RQ1 model | `fct_rq1_model_estimates`, `fct_rq1_year_effects` |
| 9 | **Wallet vs "the economy"** bars with bootstrap CIs | RQ3 | `fct_pocketbook_gap*` |
| 10 | **Robustness battery** for the two headline numbers | "Would a different choice change the answer?" | RQ1 + RQ3 marts |
| 11 | **Expected vs actual inflation**, salience test, correlation grid | RQ2 | `fct_expected_vs_actual_inflation*`, `fct_rq2_correlations` |
| 12 | **Say vs do** | RQ4 (context for the Imprint extension) | `fct_say_vs_do_*` |
| 13 | **Further analysis** not in the marts: asymmetry test, baseline/endpoint sensitivity, Holm adjustment, cross-source concordance, the record-wide gap | Panel questions | marts only |
| 14 | **Story numbers** wired into a draft nut graf | Raw data → prose | number sheet |
| 15 | **Exports** + session info | Reproducibility | — |

## Rules this notebook follows
1. **No new estimates hide in a chart.** Every plotted value is a column in a mart. Section 13 computes a few new
   quantities (all from marts, no microdata), and each is labeled *exploratory* or *robustness* where it is made.
2. **Symmetric framing.** Parties keep fixed colors (Republican red, Democrat blue, independents amber), and every chart
   that shows one party's swing shows the other's. Titles describe patterns, never judge an administration.
3. **Uncertainty is drawn, not footnoted.** Bootstrap intervals, noise bands and the CES quiet-year yardstick appear on
   the charts they qualify.
4. **Levels are never compared across sources** (assumption A4): UMich, CES and Pew sit in separate panels.
5. **Every number in the prose is a variable** (section 14), so a revision upstream cannot leave a stale number in the draft.

## Run order
Notebooks 01 → 02 → 03 → 04 → **05**. Nothing here writes back to the schema; outputs (PNG/SVG charts, CSV tables) go to
a volume (Databricks) or a local `outputs/` folder. Runtime: under a minute.

# 0. Setup

In [0]:
# ── Standard library ─────────────────────────────────────────────────────────
import os
import textwrap
import warnings
from datetime import datetime, timezone
from pathlib import Path

# ── Analysis stack (all pre-installed on Databricks serverless) ──────────────
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from scipy import stats
%pip install statsmodels
from statsmodels.stats.multitest import multipletests
from IPython.display import display, Markdown

warnings.filterwarnings("ignore", category=FutureWarning)   # pandas groupby chatter; never hides a result

# ── Where the tables live ────────────────────────────────────────────────────
CATALOG = "workspace"
SCHEMA = "public_opinion_economy_case_study"

# `spark` exists in every Databricks notebook. If it does not, we are running locally and read parquet instead.
try:
    spark  # noqa: F821
    IN_DATABRICKS = True
except NameError:
    IN_DATABRICKS = False

LOCAL_MARTS_DIR = Path(os.environ.get("LOCAL_MARTS_DIR", "marts"))   # one parquet file or folder per table

# ── Where the outputs go ─────────────────────────────────────────────────────
# Databricks: a dedicated volume, so charts are downloadable from Catalog Explorer and never mixed with raw data.
# Local: ./outputs next to the notebook (figures/ and tables/ underneath).
SAVE_OUTPUTS = True
if IN_DATABRICKS:
    spark.sql(f"USE CATALOG {CATALOG}")                                   # noqa: F821
    spark.sql(f"USE SCHEMA {SCHEMA}")                                     # noqa: F821
    spark.sql("CREATE VOLUME IF NOT EXISTS outputs COMMENT 'Charts and tables exported by notebook 05'")  # noqa: F821
    OUTPUT_DIR = Path(f"/Volumes/{CATALOG}/{SCHEMA}/outputs")
else:
    OUTPUT_DIR = Path(os.environ.get("OUTPUT_DIR", "outputs"))
FIG_DIR, TABLE_DIR = OUTPUT_DIR / "figures", OUTPUT_DIR / "tables"
if SAVE_OUTPUTS:
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    TABLE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Running in {'Databricks' if IN_DATABRICKS else 'local mode'}; outputs -> {OUTPUT_DIR}")

### 0a · Chart style
One style for every figure, so the deck reads as one system.
* **Party colors are semantic and fixed** (color follows the entity, never its rank). The three hues were run through a
  colorblind validator (all-pairs CVD ΔE ≥ 10, normal-vision ΔE ≥ 16, contrast ≥ 3:1 on white): Republican red,
  Democrat blue, independent **amber** (not purple, which collapses into blue under protanopia).
* "All respondents" and anything non-partisan (fundamentals, inflation) are drawn in **neutral ink**, so color always
  means party.
* Every multi-series chart has a legend *and* direct labels, so identity never rests on color alone.
* One y-axis per panel, always. Quantities in different units go in separate panels, never on a twin axis.

In [0]:
# ── Palette ──────────────────────────────────────────────────────────────────
COLORS = {
    "Republican": "#B8322A",
    "Democrat": "#2F6FB3",
    "Independent": "#B07D12",
    "All respondents": "#4A4A4A",
}
INK, MUTED, GRID, BAND = "#1F1F1F", "#6B6B6B", "#E4E4E4", "#D9D9D9"
TINT = {"Republican": "#B8322A", "Democrat": "#2F6FB3"}           # used at very low alpha for White House shading
PARTY_SERIES = ["Republican", "Democrat", "Independent"]

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 200,
    "font.size": 10,
    "font.family": "DejaVu Sans",          # ships with matplotlib everywhere, so local and Databricks renders match
    "axes.edgecolor": MUTED,
    "axes.labelcolor": INK,
    "axes.titlesize": 11,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.color": GRID,
    "grid.linewidth": 0.8,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
    "legend.frameon": False,
    "lines.linewidth": 2,
})

SOURCE_UMICH = "University of Michigan Surveys of Consumers (index by party; 1966 = 100)"
SOURCE_PEW = "Pew Research Center (% rating national economy excellent/good; leaners with parties)"
SOURCE_CES = "Cooperative Election Study cumulative file, weighted"
SOURCE_ANES = "ANES Time Series Cumulative Data File, weighted"


def add_titles(fig, title, subtitle=None, width=115):
    """Newsroom-style header: a takeaway title and a plain-language subtitle, left-aligned above the plot area."""
    fig.text(0.01, 0.995, title, ha="left", va="top", fontsize=14, fontweight="bold", color=INK)
    if subtitle:
        fig.text(0.01, 0.945, textwrap.fill(subtitle, width), ha="left", va="top", fontsize=10, color=MUTED,
                 linespacing=1.35)


def add_source(fig, text, width=150):
    """Source and method line at the bottom-left; every chart carries one so it can travel on its own."""
    fig.text(0.01, 0.005, textwrap.fill(text, width), ha="left", va="bottom", fontsize=7.5, color=MUTED,
             linespacing=1.3)


def layout(fig, top=0.84, bottom=0.09):
    """Leave room for the header and the source line, then let matplotlib pack the panels."""
    fig.tight_layout(rect=[0, bottom, 1, top])


def save(fig, name):
    """Save PNG (slides) and SVG (web/print, editable) versions with the same stem, then display."""
    if SAVE_OUTPUTS:
        for ext in ("png", "svg"):
            fig.savefig(FIG_DIR / f"{name}.{ext}", bbox_inches="tight", facecolor="white")
    plt.show()


def shade_white_house(ax, periods, y_text=None, label=True):
    """Light background tint for who held the White House. `periods` = list of (start, end, party)."""
    for start, end, party in periods:
        ax.axvspan(start, end, color=TINT[party], alpha=0.06, lw=0, zorder=0)
        # Label only periods long enough to hold the text (a few weeks of Obama at the left edge would clip)
        long_enough = (end - start).days > 300 if hasattr(end - start, "days") else (end - start) > 0.9
        if label and y_text is not None and long_enough:
            mid = start + (end - start) / 2
            ax.text(mid, y_text, f"{party[:3]}. White House", ha="center", va="top", fontsize=7.5, color=MUTED)


def direct_label(ax, x, y, text, color, dx=0, dy=0, **kw):
    """Direct label at the end of a line. Party names are drawn in the party's color (they ARE the identity key, doubling
    the legend); numbers elsewhere stay in neutral ink."""
    ax.annotate(text, (x, y), xytext=(dx, dy), textcoords="offset points", color=color, fontsize=9,
                fontweight="bold", va="center", **kw)

## 1. Load every mart once
All 19 tables are read into a dict `M`, so every section below works from the same snapshot of the data. **Monthly marts grow when new UMich/CPI months are added**, so a
mismatch is reported as a warning, not an error; the number sheet in section 2 is the real check.

In [0]:
EXPECTED_ROWS = {
    # SQL marts (notebook 03)
    "fct_party_ratings": 188, "fct_wave_contrasts": 138, "fct_sentiment_monthly": 640, "fct_event_study": 482,
    "fct_swing_timing": 21, "fct_fundamentals_at_transitions": 33, "fct_midterm_placebo": 21, "fct_pocketbook_gap": 12,
    "fct_expected_vs_actual_inflation": 128, "fct_expected_vs_actual_inflation_by_year": 12, "fct_rq2_annual_panel": 30,
    "fct_say_vs_do_monthly": 128,
    # Python model tables (notebook 04)
    "fct_rq1_model_estimates": 32, "fct_rq1_year_effects": 10, "fct_swing_timing_bootstrap": 21,
    "fct_pocketbook_gap_bootstrap": 12, "fct_pocketbook_gap_model": 18, "fct_rq2_correlations": 64,
    "fct_say_vs_do_correlations": 16,
}

# Columns that hold dates. Spark DATE arrives in pandas as python `datetime.date` objects; converting them to
# Timestamps once here means every chart can use them on a time axis without per-chart fixes.
DATE_COLUMNS = {"month", "wave_date", "date_t0", "date_t1", "election_date", "election_month", "inauguration_month",
                "peak_expected_median_month", "first_month", "last_month", "fieldwork_month", "window_start_month",
                "window_end_month"}


def load(name: str) -> pd.DataFrame:
    """Read one mart into pandas from Unity Catalog (Databricks) or a local parquet copy."""
    if IN_DATABRICKS:
        pdf = spark.table(f"{CATALOG}.{SCHEMA}.{name}").toPandas()                       # noqa: F821
    else:
        path = LOCAL_MARTS_DIR / name
        path = path if path.exists() else LOCAL_MARTS_DIR / f"{name}.parquet"
        pdf = pd.read_parquet(path)
    for col in DATE_COLUMNS.intersection(pdf.columns):
        # Guard: in fct_fundamentals_at_transitions, `election_month` / `inauguration_month` are NUMERIC checkpoint
        # values (e.g. inflation in the inauguration month), not dates. Only convert columns that are not numbers.
        if not pd.api.types.is_numeric_dtype(pdf[col]):
            pdf[col] = pd.to_datetime(pdf[col])
    return pdf


M = {name: load(name) for name in EXPECTED_ROWS}

load_report = pd.DataFrame([{"table": n, "rows": len(df), "expected_rows": EXPECTED_ROWS[n],
                             "matches_handoff": len(df) == EXPECTED_ROWS[n]} for n, df in M.items()])
display(load_report)
if not load_report["matches_handoff"].all():
    print("WARNING: row counts differ from the handoff. Expected if new months were loaded; check the number sheet.")

# Short aliases for the tables used most (purely for readability below)
ratings, contrasts = M["fct_party_ratings"], M["fct_wave_contrasts"]
monthly, es = M["fct_sentiment_monthly"], M["fct_event_study"]
swing_boot, fund = M["fct_swing_timing_bootstrap"], M["fct_fundamentals_at_transitions"]
placebo, rq1 = M["fct_midterm_placebo"], M["fct_rq1_model_estimates"]
pb, pb_boot, pb_model = M["fct_pocketbook_gap"], M["fct_pocketbook_gap_bootstrap"], M["fct_pocketbook_gap_model"]

## 2. The number sheet: every figure the story quotes, traced to a table
This is the fact-check layer the job description asks for ("retrospectively defensible if and when questioned"). Each
row is one number that appears in the slides or the draft, with:
* the **mart and filter** it comes from (so a reviewer can re-run it in one SQL line),
* the **value read live** from the mart, and
* the **value verified** (local Spark run, 22 Oct), with a pass/fail at display precision.

If an upstream fix changes a number, this table is where it shows up first, and section 14 picks up the new value
automatically. Helper functions keep each lookup to one readable line.

In [0]:
def one(df: pd.DataFrame, col: str, **filters):
    """Return exactly one value from `df[col]` after equality filters; fail loudly if the filter is not unique.
    A silent `.iloc[0]` on a non-unique filter is how wrong numbers get into print."""
    sub = df
    for k, v in filters.items():
        sub = sub[sub[k] == v]
    if len(sub) != 1:
        raise ValueError(f"{col} with {filters}: expected 1 row, found {len(sub)}")
    return sub[col].iloc[0]


def ces_pct(year, party, col="pct_better", definition="Leaners with parties"):
    return one(ratings, col, source="CES", survey_year=year, party_group=party, party_definition=definition)


def pew_contrast(event, window, col):
    return one(contrasts, col, source="Pew", event_key=event, comparison_window=window, measure="pct_excellent_good")


def swing(event, series, col):
    return one(swing_boot, col, event_key=event, series=series)


PEW_6M = "Pre-election to six months after inauguration"
PEW_BRACKET = "Last poll before vs first poll after inauguration"

# The RQ1 premium specs that share the main outcome and unit (percentage points of "better"), i.e. the robustness
# range quoted in the story. Mirror outcome (worse), 5-point score and logit are reported separately.
PP_SPECS = ["main", "ns_as_same", "leaners_as_independents", "no_controls", "excluding_covid",
            "year_specific_weights", "unweighted"]
premium = rq1[(rq1["term"] == "co_partisan") & rq1["spec"].isin(PP_SPECS)]

infl = fund[fund["measure"] == "Inflation (CPI, % y/y)"]
party_sent = fund[fund["measure"].isin(["Sentiment: winning party (UMich index)", "Sentiment: losing party (UMich index)"])
                  & fund["event_key"].isin(["pres-2020", "pres-2024"])]
evi = M["fct_expected_vs_actual_inflation_by_year"]
yearly_gap = (monthly[(monthly["series"] == "Republican minus Democrat") & monthly["sentiment_index"].notna()]
              .groupby("year")["sentiment_index"].mean())
sdc = M["fct_say_vs_do_correlations"]
pbm_b3 = pb_model[pb_model["term"] == "national_minus_personal (b3)"]

# (id, claim as it appears in the story, value, handoff value, decimals for the comparison, source pointer)
NUMBERS = [
    ("ces_rep_2024", "CES: Republicans saying economy got better, fall 2024 (%)", ces_pct(2024, "Republican"), 5.0, 1,
     "fct_party_ratings · CES, Leaners with parties, main spec"),
    ("ces_rep_2025", "CES: Republicans saying economy got better, fall 2025 (%)", ces_pct(2025, "Republican"), 52.2, 1, "same"),
    ("ces_dem_2024", "CES: Democrats saying economy got better, fall 2024 (%)", ces_pct(2024, "Democrat"), 52.5, 1, "same"),
    ("ces_dem_2025", "CES: Democrats saying economy got better, fall 2025 (%)", ces_pct(2025, "Democrat"), 5.1, 1, "same"),
    ("ces_ind_2024", "CES: independents, fall 2024 (%)", ces_pct(2024, "Independent"), 11.7, 1, "same"),
    ("ces_ind_2025", "CES: independents, fall 2025 (%)", ces_pct(2025, "Independent"), 13.1, 1, "same"),
    ("ces_ind_2016", "CES: independents, fall 2016 (%)", ces_pct(2016, "Independent"), 16.0, 1, "same"),
    ("ces_ind_2017", "CES: independents, fall 2017 (%)", ces_pct(2017, "Independent"), 42.5, 1, "same"),
    ("pew_rep_t0_24", "Pew: Republicans, last pre-election poll 2024 (%)", pew_contrast("pres-2024", PEW_6M, "rep_t0"), 10, 0,
     "fct_wave_contrasts · Pew, pre-election → six months window"),
    ("pew_rep_t1_24", "Pew: Republicans, ~six months after inauguration 2025 (%)", pew_contrast("pres-2024", PEW_6M, "rep_t1"), 36, 0, "same"),
    ("pew_dem_t0_24", "Pew: Democrats, last pre-election poll 2024 (%)", pew_contrast("pres-2024", PEW_6M, "dem_t0"), 41, 0, "same"),
    ("pew_dem_t1_24", "Pew: Democrats, ~six months after inauguration 2025 (%)", pew_contrast("pres-2024", PEW_6M, "dem_t1"), 11, 0, "same"),
    ("rq1_premium", "Co-partisan premium, main spec (pp)", one(rq1, "estimate", spec="main", term="co_partisan"), 40.0, 1,
     "fct_rq1_model_estimates · spec=main, term=co_partisan"),
    ("rq1_ci_low", "Co-partisan premium, 95% CI low (state-clustered)", one(rq1, "ci_low_cluster", spec="main", term="co_partisan"), 39.3, 1, "same"),
    ("rq1_ci_high", "Co-partisan premium, 95% CI high", one(rq1, "ci_high_cluster", spec="main", term="co_partisan"), 40.7, 1, "same"),
    ("rq1_n", "Co-partisan premium, respondents in model", one(rq1, "n_obs", spec="main", term="co_partisan"), 383533, 0, "same"),
    ("rq1_rep", "Premium for Republicans (split model, pp)", one(rq1, "estimate", spec="split_by_party", term="co_partisan_rep"), 38.2, 1,
     "fct_rq1_model_estimates · spec=split_by_party"),
    ("rq1_dem", "Premium for Democrats (split model, pp)", one(rq1, "estimate", spec="split_by_party", term="co_partisan_dem"), 41.8, 1, "same"),
    ("rq1_min", "Premium, lowest across same-unit specs (pp)", premium["estimate"].min(), 38.9, 1, f"min over specs {PP_SPECS}"),
    ("rq1_max", "Premium, highest across same-unit specs (pp)", premium["estimate"].max(), 42.2, 1, "max over same specs"),
    ("rq1_logit", "Logit average marginal effect (pp)", one(rq1, "estimate", spec="logit_ame", term="co_partisan"), 36.9, 1, "spec=logit_ame"),
    ("ye_2017", "Year effect 2017 vs 2016 (pp, common to everyone)",
     one(M["fct_rq1_year_effects"], "year_effect_pp", survey_year=2017), 20.8, 1, "fct_rq1_year_effects"),
    ("sw_rep20_t", "Swing done at inauguration, Republicans 2020–21 (%)", swing("pres-2020", "Republican", "pct_swing_done_at_takeover"), 59, 0,
     "fct_swing_timing_bootstrap"),
    ("sw_dem20_t", "Swing done at inauguration, Democrats 2020–21 (%)", swing("pres-2020", "Democrat", "pct_swing_done_at_takeover"), 56, 0, "same"),
    ("sw_rep24_e", "Swing done at election, Republicans 2024–25 (%)", swing("pres-2024", "Republican", "pct_swing_done_at_election"), 16, 0, "same"),
    ("sw_rep24_t", "Swing done at inauguration, Republicans 2024–25 (%)", swing("pres-2024", "Republican", "pct_swing_done_at_takeover"), 70, 0, "same"),
    ("sw_dem24_e", "Swing done at election, Democrats 2024–25 (%)", swing("pres-2024", "Democrat", "pct_swing_done_at_election"), 0, 0, "same"),
    ("sw_dem24_t", "Swing done at inauguration, Democrats 2024–25 (%)", swing("pres-2024", "Democrat", "pct_swing_done_at_takeover"), 35, 0, "same"),
    ("sw_rep24_tlo", "  95% interval low (Rep 2024–25 at inauguration)", swing("pres-2024", "Republican", "takeover_ci_low"), 60, 0, "same"),
    ("sw_rep24_thi", "  95% interval high", swing("pres-2024", "Republican", "takeover_ci_high"), 81, 0, "same"),
    ("infl_sep24", "Inflation, Sep 2024 (%)", one(infl, "sep_before", event_key="pres-2024"), 2.4, 1, "fct_fundamentals_at_transitions"),
    ("infl_jan25", "Inflation, Jan 2025 (%)", one(infl, "inauguration_month", event_key="pres-2024"), 3.0, 1, "same"),
    ("infl_jul21", "Inflation, Jul 2021 (%)", one(infl, "six_months_after", event_key="pres-2020"), 5.4, 1, "same"),
    ("sent_move_min", "Smallest party sentiment move, Sep → inauguration month (index pts, abs)",
     party_sent["change_sep_to_inauguration"].abs().min(), 17, 0, "same, perception rows"),
    ("sent_move_max", "Largest party sentiment move, Sep → inauguration month (index pts, abs)",
     party_sent["change_sep_to_inauguration"].abs().max(), 30, 0, "same"),
    ("mid22_gap", "UMich: Rep–Dem gap change 3 months after 2022 midterm (pts)",
     one(placebo, "rep_minus_dem_gap_change", source="UMich (monthly)", event_key="mid-2022", checkpoint_months_from_election=3), 2.2, 1,
     "fct_midterm_placebo · UMich, +3 months"),
    ("mid18_gap", "UMich: Rep–Dem gap change 3 months after 2018 midterm (pts)",
     one(placebo, "rep_minus_dem_gap_change", source="UMich (monthly)", event_key="mid-2018", checkpoint_months_from_election=3), -3.1, 1, "same"),
    ("rq3_ratio16", "ANES: national ÷ own-finances gap, 2016 (main spec, ×)", one(pb_boot, "national_to_personal_ratio", survey_year=2016, is_main_spec=True), 2.29, 2,
     "fct_pocketbook_gap_bootstrap · main spec"),
    ("rq3_ratio20", "ANES: national ÷ own-finances gap, 2020 (×)", one(pb_boot, "national_to_personal_ratio", survey_year=2020, is_main_spec=True), 3.56, 2, "same"),
    ("rq3_ratio24", "ANES: national ÷ own-finances gap, 2024 (×)", one(pb_boot, "national_to_personal_ratio", survey_year=2024, is_main_spec=True), 1.99, 2, "same"),
    ("rq3_nat24", "ANES: national-economy gap 2024, 'do not know' in base (pts)",
     one(pb, "national_gap_pts", survey_year=2024, outcome_spec="Robustness: do not know in base", party_definition="Leaners with parties"), 50.3, 1,
     "fct_pocketbook_gap"),
    ("rq3_b3_16", "ANES: national minus personal gap with controls, 2016 (pp)",
     one(pbm_b3, "estimate_pp", survey_year=2016, with_controls=True), 16.4, 1, "fct_pocketbook_gap_model · b3"),
    ("rq3_b3_20", "ANES: national minus personal gap with controls, 2020 (pp)", one(pbm_b3, "estimate_pp", survey_year=2020, with_controls=True), 28.3, 1, "same"),
    ("rq3_b3_24", "ANES: national minus personal gap with controls, 2024 (pp)", one(pbm_b3, "estimate_pp", survey_year=2024, with_controls=True), 25.0, 1, "same"),
    ("exp_2022", "Median expected minus actual inflation, 2022 avg (pts)", one(evi, "avg_expected_minus_current_pct", period="2022"), -3.0, 1,
     "fct_expected_vs_actual_inflation_by_year"),
    ("exp_2025", "Median expected minus actual inflation, 2025 avg (pts)", one(evi, "avg_expected_minus_current_pct", period="2025"), 2.2, 1, "same"),
    ("exp_peak", "Peak median expected inflation, 2025 (%)", one(evi, "peak_expected_median_pct", period="2025"), 6.6, 1, "same"),
    ("gap_2025", "UMich Rep–Dem gap, 2025 average (index pts)", yearly_gap.get(2025, np.nan), 45.5, 1, "fct_sentiment_monthly · yearly mean"),
    ("gap_2026", "UMich Rep–Dem gap, 2026 average so far (index pts)", yearly_gap.get(2026, np.nan), 52.8, 1, "same"),
    ("r_all_pce", "Corr(all-respondents sentiment change, real PCE growth), all months",
     one(sdc, "r", sample="All months", say_measure="All respondents' sentiment", do_measure="Real PCE growth (% m/m)"), 0.42, 2,
     "fct_say_vs_do_correlations"),
    ("r_all_pce_x", "  … excluding 2020–21",
     one(sdc, "r", sample="Excluding 2020-2021 (plan robustness)", say_measure="All respondents' sentiment", do_measure="Real PCE growth (% m/m)"), 0.08, 2, "same"),
    ("r_party_max", "Largest |corr| of a party's sentiment change with real PCE growth",
     sdc[sdc["say_measure"] != "All respondents' sentiment"].query("do_measure == 'Real PCE growth (% m/m)'")["r"].abs().max(), 0.10, 2, "same"),
]

number_sheet = pd.DataFrame(NUMBERS, columns=["id", "claim", "value", "handoff_value", "decimals", "source"])
number_sheet["value"] = number_sheet["value"].astype(float)
# Pass = equal at the precision the number is quoted (one decimal for most shares, whole numbers for swing shares).
number_sheet["matches_handoff"] = [round(v, int(d)) == round(h, int(d)) or abs(v - h) <= 0.5 * 10 ** (-int(d)) + 1e-9
                                   for v, h, d in zip(number_sheet["value"], number_sheet["handoff_value"], number_sheet["decimals"])]
number_sheet["display"] = [f"{v:,.{int(d)}f}" for v, d in zip(number_sheet["value"], number_sheet["decimals"])]
N = dict(zip(number_sheet["id"], number_sheet["value"]))   # N["rq1_premium"] -> 40.02...; used in section 14

print(f"{number_sheet['matches_handoff'].sum()} of {len(number_sheet)} numbers match the handoff at display precision.")
display(number_sheet[["id", "claim", "display", "handoff_value", "matches_handoff", "source"]])
if not number_sheet["matches_handoff"].all():
    display(Markdown("**⚠ Some numbers moved since the handoff.** Check the rows marked False before quoting them; "
                     "if the change is intended (new data, a fix), update the plan and the deck from this table."))

## 3. Lede chart: the flip, month by month
**What it shows:** UMich's monthly sentiment index for Republicans, Democrats and independents, April 2017 to the latest
month, with presidential elections (dashed), inaugurations (solid) and midterms (dotted) marked and the White House party
tinted in the background. Pew's polls sit in a **separate panel** because they ask a different question (is the economy
excellent/good *now*), so only their direction and timing are comparable (assumption A4).

**Why this form:** a line chart over time is the one form where the flip is visible "without a word of explanation"
(the plan's test for the lede). Direct labels at the line ends replace the need to look at the legend.

In [0]:
lede = monthly[monthly["series"].isin(PARTY_SERIES) & monthly["sentiment_index"].notna()].copy()
pew = ratings[(ratings["source"] == "Pew") & ratings["party_group"].isin(["Republican", "Democrat"])].copy()
markers = monthly.dropna(subset=["event_marker"]).drop_duplicates("month")[["month", "event_marker", "event_marker_type"]]

# White House periods from the monthly table (contiguous runs of the same party), clipped to the chart window
x_start, x_end = pd.Timestamp("2016-10-01"), lede["month"].max() + pd.offsets.MonthBegin(1)
wh = monthly.drop_duplicates("month").sort_values("month")[["month", "white_house_party"]].dropna()
wh["run"] = (wh["white_house_party"] != wh["white_house_party"].shift()).cumsum()
wh_periods = [(max(g["month"].min(), x_start), min(g["month"].max() + pd.offsets.MonthBegin(1), x_end), g["white_house_party"].iloc[0])
              for _, g in wh.groupby("run")]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8.2), sharex=True, gridspec_kw={"height_ratios": [2.3, 1]})

# ── Top panel: UMich party lines ─────────────────────────────────────────────
shade_white_house(ax1, wh_periods, y_text=lede["sentiment_index"].max() + 9)
for s in PARTY_SERIES:
    d = lede[lede["series"] == s].sort_values("month")
    ax1.plot(d["month"], d["sentiment_index"], color=COLORS[s], lw=2, label=s)
    direct_label(ax1, d["month"].iloc[-1], d["sentiment_index"].iloc[-1], s, COLORS[s], dx=6)

# Event lines: style encodes the event type (dashed = election, solid = inauguration, dotted = midterm)
EVENT_STYLE = {"Presidential election": dict(ls="--", lw=1.1), "Inauguration": dict(ls="-", lw=1.1),
               "Midterm election": dict(ls=":", lw=1.1)}
for ax in (ax1, ax2):
    for _, ev in markers[(markers["month"] >= x_start) & (markers["month"] <= x_end)].iterrows():
        ax.axvline(ev["month"], color=MUTED, alpha=0.7, zorder=1, **EVENT_STYLE[ev["event_marker_type"]])
for _, ev in markers[(markers["month"] >= x_start) & (markers["event_marker_type"] != "Midterm election")].iterrows():
    short = ev["event_marker"].split(" (")[0].replace(" presidential election", " election")
    ax1.text(ev["month"], lede["sentiment_index"].min() - 6, short, rotation=90, fontsize=7.5, color=MUTED,
             ha="right", va="bottom")

ax1.set_ylabel("Sentiment index (1966 = 100)")
ax1.set_ylim(lede["sentiment_index"].min() - 8, lede["sentiment_index"].max() + 11)
ax1.set_title("UMich consumer sentiment, monthly, by party", fontsize=10.5)
# Legend sits above the plot area, right-aligned on the panel-title line, so it never covers data or the tint labels
ax1.legend(handles=[Line2D([], [], color=COLORS[s], lw=2, label=s) for s in PARTY_SERIES]
           + [Line2D([], [], color=MUTED, ls="--", lw=1.1, label="Presidential election"),
              Line2D([], [], color=MUTED, ls="-", lw=1.1, label="Inauguration"),
              Line2D([], [], color=MUTED, ls=":", lw=1.1, label="Midterm")],
           ncol=3, loc="lower right", bbox_to_anchor=(1.0, 1.0), fontsize=8.5)

# ── Bottom panel: Pew cross-check (different question, so its own axis) ──────
for s in ["Republican", "Democrat"]:
    d = pew[pew["party_group"] == s].sort_values("wave_date")
    ax2.plot(d["wave_date"], d["pct_positive"], color=COLORS[s], lw=1.6, marker="o", ms=4.5,
             markeredgecolor="white", markeredgewidth=0.8)
    direct_label(ax2, d["wave_date"].iloc[-1], d["pct_positive"].iloc[-1], s, COLORS[s], dx=6)
shade_white_house(ax2, wh_periods, label=False)
ax2.set_ylabel("% excellent/good")
ax2.set_ylim(0, max(70, pew["pct_positive"].max() + 8))
ax2.set_title("Pew: share rating the economy excellent or good, by party (each dot is a poll)", fontsize=10.5)
ax2.xaxis.set_major_locator(mdates.YearLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax2.set_xlim(x_start, x_end + pd.DateOffset(months=7))     # room for the direct labels

add_titles(fig, "When the White House changes hands, each party's view of the economy flips",
           "Republicans and Democrats trade places within months of each presidential election, in two independent "
           "surveys. Independents move far less. The gap between the parties is now the widest in the series.")
add_source(fig, f"Sources: {SOURCE_UMICH}; {SOURCE_PEW}. UMich party series begins April 2017. The two panels ask different "
                "questions, so compare direction and timing, not levels. Tint = party holding the White House.")
layout(fig, top=0.88, bottom=0.07)
save(fig, "01_lede_monthly_party_sentiment")

## 4. Event study: when does opinion actually flip?
**What it shows:** each party's change in sentiment from its July–September baseline, month by month from six months
before the election to fourteen after, for the two transitions the UMich party series covers. The gray band is the
**noise band** (±2 standard errors of a change from baseline, using the noisiest of the three party series, from the
robust month-to-month noise model in `fct_event_study`). Anything inside it is indistinguishable from an ordinary month.

**The co-primary timing estimand** (plan revision, 22 Oct) is printed on each panel: the share of each party's total swing
(baseline → six months after inauguration) that was complete in the election month and in the inauguration month, with
the noise-bootstrap 95% interval from notebook 04. Both dates are reported whatever they show, because the election date
was added after seeing the data.

In [0]:
EVENTS_UMICH = [("pres-2020", "2020 election → 2021 inauguration"), ("pres-2024", "2024 election → 2025 inauguration")]

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.6), sharey=True)
for ax, (ev, ev_title) in zip(axes, EVENTS_UMICH):
    d = es[(es["event_key"] == ev) & es["series"].isin(PARTY_SERIES)]
    band = d.groupby("series")["noise_band_2se"].max().max()            # widest party band: conservative shading
    ax.axhspan(-band, band, color=BAND, alpha=0.6, lw=0, zorder=0)
    ax.axhline(0, color=MUTED, lw=0.8)

    for s in PARTY_SERIES:
        dd = d[d["series"] == s].sort_values("months_from_election")
        ax.plot(dd["months_from_election"], dd["change_from_baseline"], color=COLORS[s], lw=2, marker="o", ms=3.5,
                markeredgecolor="white", markeredgewidth=0.6, label=s)
        direct_label(ax, dd["months_from_election"].iloc[-1], dd["change_from_baseline"].iloc[-1], s, COLORS[s], dx=5)

    # The two co-primary dates and the swing endpoint
    # Event labels sit just ABOVE the plot area (the panel title is lifted with `pad`), so they never cross a line
    for x, lab, ls in [(0, "Election", "--"), (2, "Inaug.", "-"), (8, "+6 mo. after (swing endpoint)", ":")]:
        ax.axvline(x, color=MUTED, ls=ls, lw=1.1, zorder=1)
        ax.annotate(lab, (x, 1.0), xycoords=("data", "axes fraction"), xytext=(0, 3), textcoords="offset points",
                    fontsize=8, color=MUTED, va="bottom", ha="center")

    # Swing-timing estimand with bootstrap intervals, read from fct_swing_timing_bootstrap.
    # Printed BELOW the panel (under the x-axis label) so it can never cover a line.
    lines = []
    for s in ["Republican", "Democrat"]:
        r = swing_boot[(swing_boot["event_key"] == ev) & (swing_boot["series"] == s)].iloc[0]
        lines.append(f"{s}s: {r['pct_swing_done_at_election']:.0f}% done by election "
                     f"({r['election_ci_low']:.0f}–{r['election_ci_high']:.0f}), "
                     f"{r['pct_swing_done_at_takeover']:.0f}% by inauguration ({r['takeover_ci_low']:.0f}–{r['takeover_ci_high']:.0f})")
    ax.text(0.0, -0.24, "\n".join(lines), transform=ax.transAxes, fontsize=8, color=INK, va="top",
            bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor=GRID))

    ax.set_title(ev_title, fontsize=10.5, pad=18)
    ax.set_xlabel("Months from the presidential election")
    ax.set_xticks(range(-6, 15, 2))
    ax.set_xlim(-6.3, 16.5)
axes[0].set_ylabel("Change from Jul–Sep baseline (index points)")
# Noise band explained once, in an empty corner of the first panel (lower left: before the election nothing moves)
axes[0].legend(handles=[Patch(color=BAND, label="Noise band (±2 SE): an ordinary month")], loc="lower left", fontsize=8)

add_titles(fig, "The flip starts around Election Day and keeps going for months after the inauguration",
           "Change in each party's sentiment from its pre-election baseline. The winning party's supporters grow more upbeat and the "
           "losing party's more downbeat; a third to two-thirds of each swing is done by the inauguration month.")
add_source(fig, f"Source: {SOURCE_UMICH}. Baseline = mean of July–September before the election. Swing = change from baseline "
                "six months after the inauguration. Intervals: parametric bootstrap of monthly measurement noise (5,000 draws). "
                "Two transitions only, so read as 'in both cases', not 'on average'.")
# Explicit margins here (not tight_layout): the swing-timing boxes live below the axes, which tight_layout ignores.
fig.subplots_adjust(left=0.07, right=0.93, top=0.76, bottom=0.31, wspace=0.12)
save(fig, "02_event_study_party_sentiment")

### 4a · Swing timing with intervals (the estimand on its own)
The same estimand as a dot-and-interval chart: hollow dots = share of the swing complete in the election month, filled
dots = in the inauguration month, lines = 95% bootstrap intervals. Rows with no interpretable share (independents in
2024–25 reversed direction) are left out by the pre-specified rule, not by choice, and are listed under the chart.

In [0]:
ROWS = [("pres-2020", "Republican"), ("pres-2020", "Democrat"), ("pres-2020", "Winning minus losing party"),
        ("pres-2024", "Republican"), ("pres-2024", "Democrat"), ("pres-2024", "Winning minus losing party")]
LABEL_EVENT = {"pres-2016": "2016–17", "pres-2020": "2020–21", "pres-2024": "2024–25"}

fig, ax = plt.subplots(figsize=(10.5, 4.8))
yticks, ylabels = [], []
for i, (ev, s) in enumerate(ROWS[::-1]):
    r = swing_boot[(swing_boot["event_key"] == ev) & (swing_boot["series"] == s)].iloc[0]
    color = COLORS.get(s, INK)
    ax.plot([r["election_ci_low"], r["election_ci_high"]], [i + 0.12] * 2, color=color, lw=1.4, alpha=0.6)
    ax.plot([r["takeover_ci_low"], r["takeover_ci_high"]], [i - 0.12] * 2, color=color, lw=2.6)
    ax.plot(r["pct_swing_done_at_election"], i + 0.12, "o", ms=8, mfc="white", mec=color, mew=1.8)
    ax.plot(r["pct_swing_done_at_takeover"], i - 0.12, "o", ms=8, color=color, mec="white", mew=1)
    ax.text(r["pct_swing_done_at_takeover"], i - 0.42, f"{r['pct_swing_done_at_takeover']:.0f}%", ha="center",
            fontsize=7.5, color=INK)
    yticks.append(i)
    ylabels.append(f"{LABEL_EVENT[ev]} · {'Gap (winner − loser)' if s.startswith('Winning') else s + 's'}")
ax.set_yticks(yticks, ylabels)
ax.axvline(0, color=MUTED, lw=0.8)
ax.axvline(100, color=MUTED, lw=0.8, ls=":")
ax.text(100, len(ROWS) - 0.45, "swing complete", fontsize=7.5, color=MUTED, ha="center")
ax.set_xlim(-15, 110)
ax.set_xlabel("Share of the total swing complete (%)")
ax.grid(axis="x", color=GRID)
ax.grid(axis="y", visible=False)
ax.legend(handles=[Line2D([], [], marker="o", ls="", mfc="white", mec=INK, ms=8, label="Election month"),
                   Line2D([], [], marker="o", ls="", color=INK, ms=8, label="Inauguration month")],
          loc="lower right", fontsize=8.5)
add_titles(fig, "Most of the swing happens between the election and the inauguration, and it is not over by then",
           "Share of each party's six-month swing complete at the two co-primary dates, with 95% intervals.")
not_shown = swing_boot[(swing_boot["event_type"] == "Presidential election") & swing_boot["pct_swing_done_at_takeover"].isna()]
add_source(fig, f"Source: {SOURCE_UMICH}; fct_swing_timing_bootstrap. Not shown (no interpretable share under the pre-specified rule): "
                + "; ".join(f"{LABEL_EVENT.get(e, e)} {s}" for e, s in zip(not_shown["event_key"], not_shown["series"])) + ".")
layout(fig, top=0.86, bottom=0.10)
save(fig, "03_swing_timing_intervals")

## 5 · Did the economy move when opinion did?
**The table** is the plan's fundamentals-at-transitions table, read straight from the mart: inflation and unemployment
at the same four checkpoints as party sentiment. **The chart** puts the *changes* from September to the inauguration month
side by side, fundamentals in one panel (percentage points) and sentiment in the other (index points). They are in
different units, so they get separate axes rather than a twin axis; the story's comparison is "a fraction of a point vs
tens of points", which holds on any scale.

In [0]:
TABLE_MEASURES = ["Inflation (CPI, % y/y)", "Unemployment rate (derived, %)",
                  "Sentiment: winning party (UMich index)", "Sentiment: losing party (UMich index)",
                  "Sentiment: independents (UMich index)", "Sentiment: all respondents (UMich index)"]
fund_table = (fund[fund["white_house_changed_parties"] & fund["event_key"].isin(["pres-2016", "pres-2020", "pres-2024"])
                   & fund["measure"].isin(TABLE_MEASURES)]
              .sort_values(["event_key", "measure_order"])
              [["event_label", "winning_party", "measure", "sep_before", "election_month", "inauguration_month",
                "six_months_after", "change_sep_to_inauguration"]]
              .rename(columns={"event_label": "Transition", "winning_party": "Winner", "measure": "Measure",
                               "sep_before": "Sep before", "election_month": "Election month",
                               "inauguration_month": "Inauguration month", "six_months_after": "July after",
                               "change_sep_to_inauguration": "Change, Sep → inauguration"}))
display(fund_table.style.format(precision=1, na_rep="—").hide(axis="index")
        .set_caption("Fundamentals vs perceptions at each change of party (2020–21 flagged: COVID moved fundamentals too)"))
if SAVE_OUTPUTS:
    fund_table.to_csv(TABLE_DIR / "fundamentals_at_transitions.csv", index=False)

In [0]:
fig, (axL, axR) = plt.subplots(1, 2, figsize=(12, 4.6), gridspec_kw={"width_ratios": [1, 1.25]})

# Left: fundamentals (pp)
FUND_ROWS = [("Inflation (CPI, % y/y)", "Inflation"), ("Unemployment rate (derived, %)", "Unemployment")]
events = [("pres-2016", "2016–17"), ("pres-2020", "2020–21*"), ("pres-2024", "2024–25")]
y = 0
yt, yl = [], []
for ev, lab in events:
    for m, mlab in FUND_ROWS:
        v = fund[(fund["event_key"] == ev) & (fund["measure"] == m)]["change_sep_to_inauguration"]
        if len(v) and pd.notna(v.iloc[0]):
            axL.barh(y, v.iloc[0], color=COLORS["All respondents"] if m.startswith("Infl") else "#9A9A9A", height=0.7)
            axL.text(v.iloc[0] + (0.05 if v.iloc[0] >= 0 else -0.05), y, f"{v.iloc[0]:+.1f}", va="center",
                     ha="left" if v.iloc[0] >= 0 else "right", fontsize=8, color=INK)
            yt.append(y); yl.append(f"{lab} · {mlab}")
            y += 1
    y += 0.6
axL.set_yticks(yt, yl)
axL.invert_yaxis()
axL.axvline(0, color=MUTED, lw=0.8)
axL.set_xlim(-2.2, 2.2)
axL.set_xlabel("Change, September → inauguration month (percentage points)")
axL.set_title("Fundamentals barely moved", fontsize=10.5)
axL.grid(axis="x", color=GRID); axL.grid(axis="y", visible=False)

# Right: perceptions (index points); bars take the color of the actual party that won/lost
y = 0
yt, yl = [], []
for ev, lab in events:
    sub = fund[(fund["event_key"] == ev) & (fund["measure_group"] == "Perception")].sort_values("measure_order")
    winner = sub["winning_party"].iloc[0] if len(sub) else None
    loser = {"Republican": "Democrat", "Democrat": "Republican"}.get(winner)
    for _, r in sub.iterrows():
        who = {"Sentiment: winning party (UMich index)": winner, "Sentiment: losing party (UMich index)": loser,
               "Sentiment: independents (UMich index)": "Independent",
               "Sentiment: all respondents (UMich index)": "All respondents"}[r["measure"]]
        v = r["change_sep_to_inauguration"]
        axR.barh(y, v, color=COLORS[who], height=0.7)
        axR.text(v + (0.6 if v >= 0 else -0.6), y, f"{v:+.1f}", va="center", ha="left" if v >= 0 else "right",
                 fontsize=8, color=INK)
        role = r["measure"].split(": ")[1].split(" (")[0]
        yt.append(y); yl.append(f"{lab} · {role} ({who[:3]}.)" if who in ("Republican", "Democrat") else f"{lab} · {role}")
        y += 1
    y += 0.6
axR.set_yticks(yt, yl)
axR.invert_yaxis()
axR.axvline(0, color=MUTED, lw=0.8)
lim = fund[fund["measure_group"] == "Perception"]["change_sep_to_inauguration"].abs().max() + 6
axR.set_xlim(-lim, lim)
axR.set_xlabel("Change, September → inauguration month (index points)")
axR.set_title("Partisan sentiment swung by tens of points", fontsize=10.5)
axR.grid(axis="x", color=GRID); axR.grid(axis="y", visible=False)

add_titles(fig, "Inflation moved a fraction of a point. Each party's mood moved by 17 to 30.",
           "Change from September before the election to the inauguration month, for each change of party in the White House. "
           "Separate panels because the units differ.")
add_source(fig, "Sources: monthly CPI inflation; unemployment derived as misery index minus inflation (official UNRATE is a listed "
                f"data gap); {SOURCE_UMICH}. *2020–21: COVID moved fundamentals at the same time; reported but flagged. "
                "UMich party series starts April 2017, so 2016–17 shows all respondents only.")
layout(fig, top=0.85, bottom=0.12)
save(fig, "04_fundamentals_vs_perceptions")

## 6. CES: ten years of survey microdata
### 6a. Share saying the economy got better, by party, every year
The annual CES series is the backbone of the regression and the only source with a big sample every year. Error bars are
±1.96 approximate standard errors (Kish effective n), mostly smaller than the dots: with 17,000–65,000 respondents a year,
sampling error is not the issue. **Fieldwork season is** (see 6b).

In [0]:
ces = ratings[(ratings["source"] == "CES") & (ratings["party_definition"] == "Leaners with parties")
              & ratings["party_group"].isin(PARTY_SERIES)].copy()
fig, ax = plt.subplots(figsize=(11, 5))

# White House tint per survey year (party in office at fieldwork, constant within a wave; QA 13b in notebook 03)
wh_year = ces.drop_duplicates("survey_year").set_index("survey_year")["white_house_party"]
for yr, party in wh_year.items():
    ax.axvspan(yr - 0.5, yr + 0.5, color=TINT[party], alpha=0.06, lw=0, zorder=0)
for s in PARTY_SERIES:
    d = ces[ces["party_group"] == s].sort_values("survey_year")
    ax.errorbar(d["survey_year"], d["pct_better"], yerr=1.96 * d["se_pct_better_approx"], color=COLORS[s], lw=2,
                marker="o", ms=6, mec="white", mew=0.8, capsize=0, label=s)
    direct_label(ax, d["survey_year"].iloc[-1], d["pct_better"].iloc[-1], s, COLORS[s], dx=8)
for yr in (2017, 2021, 2025):
    ax.annotate("new president", (yr, 1.0), xycoords=("data", "axes fraction"), xytext=(0, -2), textcoords="offset points",
                ha="center", va="top", fontsize=7.5, color=MUTED)
ax.set_xticks(sorted(ces["survey_year"].unique()))
ax.set_xlim(2015.5, 2026.3)
ax.set_ylim(0, ces["pct_better"].max() + 12)          # headroom from the data, so no point is ever clipped
ax.set_ylabel("% saying the economy got better over the past year")
ax.legend(loc="upper right", bbox_to_anchor=(0.86, 0.94), ncol=3, fontsize=8.5)   # empty upper-right area (2021–24)
add_titles(fig, "In the CES, the parties trade places in every first year of a new presidency",
           "Share of each party saying the national economy got better over the past year ('not sure' excluded). "
           "Tint = party in the White House at fieldwork.")
add_source(fig, f"Source: {SOURCE_CES}; leaners grouped with parties. Bars = ±1.96 approximate SE (Kish effective n). Even years "
                "are fielded during the campaign (Sep–Nov), odd years after it (Nov–Dec).")
layout(fig, top=0.86, bottom=0.09)
save(fig, "05_ces_annual_by_party")

### 6b. The yardstick: how far the partisan gap moves when *nothing* happens
Found while testing the marts: in the four CES pairs with **no federal election** between waves, the Republican–Democrat
gap still drifts 12–17 points (fieldwork season, news cycle, question context). Sampling error (1–2 points) would make
almost any annual change look "significant", so this drift, not the sampling band, is the honest benchmark for an annual
comparison. Presidential transitions move the gap five to six times as far. Midterms are in between and are discussed
with the placebo (section 7).

In [0]:
drift = (contrasts[(contrasts["source"] == "CES") & (contrasts["measure"] == "pct_better")
                   & (contrasts["party_definition"] == "Leaners with parties")]
         .sort_values("date_t0").copy())
drift["pair"] = drift["wave_key_t0"].str[-4:] + "→" + drift["wave_key_t1"].str[-2:]
drift["abs_gap_change"] = drift["rep_minus_dem_gap_change"].abs()
TYPE_STYLE = {   # emphasis by type, in neutral ink (no party meaning here)
    "Presidential transition": dict(color=INK, label="Presidential transition"),
    "Midterm (a chamber of Congress changed hands)": dict(color="#8C8C8C", label="Midterm (House flipped)"),
    "No federal election between waves": dict(color="#C9C9C9", label="No federal election"),
    "Presidential re-election": dict(color="#C9C9C9", label="Re-election"),
}
fig, ax = plt.subplots(figsize=(11, 4.6))
ax.bar(drift["pair"], drift["abs_gap_change"], color=[TYPE_STYLE[t]["color"] for t in drift["comparison_type"]],
       width=0.68, edgecolor="white", linewidth=2)
for x, v in zip(drift["pair"], drift["abs_gap_change"]):
    ax.text(x, v + 1.2, f"{v:.0f}", ha="center", fontsize=8.5, color=INK)
quiet_max = drift["quiet_years_max_abs_gap_change"].dropna().iloc[0]
ax.axhline(quiet_max, color=MUTED, ls="--", lw=1)
ax.set_ylabel("Change in Rep − Dem gap (pts, absolute)")
ax.set_ylim(0, drift["abs_gap_change"].max() * 1.12)
seen = list(dict.fromkeys(TYPE_STYLE[t]["label"] for t in drift["comparison_type"]))
# Legend over the low bars between the first two transitions; the dashed line is explained in the legend, not on the plot
ax.legend(handles=[Patch(color=next(v["color"] for v in TYPE_STYLE.values() if v["label"] == lab), label=lab) for lab in seen]
          + [Line2D([], [], color=MUTED, ls="--", lw=1, label=f"Largest drift with no election ({quiet_max:.0f} pts)")],
          loc="upper left", bbox_to_anchor=(0.13, 1.0), fontsize=8.5)
add_titles(fig, "Transitions move the partisan gap five to six times as far as an ordinary year",
           "How far the Republican–Democrat gap in '% better' moved between consecutive CES waves. The dashed line is the "
           "largest move in a year with no federal election between waves.")
add_source(fig, f"Source: {SOURCE_CES}; fct_wave_contrasts (main spec, leaners with parties).")
layout(fig, top=0.84, bottom=0.08)
save(fig, "06_ces_gap_drift_yardstick")

### 6c. Common improvement vs partisan swing (new finding 1 from the handoff)
"Net of independents" splits each transition into a part everyone shared (independents' change) and a partisan part
(each party's change minus independents'). In 2016→17 independents themselves rose around 26 points, unlike 2024→25 (~1), so the
honest description of 2017 is **"common improvement plus a partisan swing"**, not partisan swing alone. Showing this
strengthens the method: the benchmark is doing its job.

In [0]:
trans = drift[drift["comparison_type"] == "Presidential transition"].copy()
fig, ax = plt.subplots(figsize=(11, 4.6))
w = 0.26
for i, (_, r) in enumerate(trans.iterrows()):
    loser = r["losing_party"]
    vals = [("Independents' change\n(shared by everyone)", r["ind_change"], COLORS["Independent"]),
            (f"Winner net of independents", r["winning_minus_ind_change"], COLORS[r["winning_party"]]),
            (f"Loser net of independents", r["losing_minus_ind_change"], COLORS[loser])]
    for j, (lab, v, c) in enumerate(vals):
        ax.bar(i + (j - 1) * w, v, width=w - 0.03, color=c)
        ax.text(i + (j - 1) * w, v + (1.5 if v >= 0 else -1.5), f"{v:+.0f}", ha="center", va="bottom" if v >= 0 else "top",
                fontsize=8.5, color=INK)
ax.axhline(0, color=MUTED, lw=0.8)
ax.set_xticks(range(len(trans)),
              [f"{r['pair']}\n(winner: {r['winning_party']}s)" for _, r in trans.iterrows()])
ax.set_ylabel("Change in % saying 'better' (pts)")
ax.set_ylim(-70, 80)
# Legend in the empty band above the bars (headroom added with the y-limit above)
ax.legend(handles=[Patch(color=COLORS["Independent"], label="Independents' change (shared component)"),
                   Patch(color="#9A9A9A", label="Winner / loser net of independents (bar color = that party)")],
          loc="upper center", ncol=2, fontsize=8.5)
add_titles(fig, "2017 was a common improvement plus a partisan swing; 2025 was almost all partisan",
           "Each transition split into the change independents shared and each party's change beyond it (CES, '% better').")
add_source(fig, f"Source: {SOURCE_CES}; fct_wave_contrasts columns ind_change, winning_minus_ind_change, losing_minus_ind_change. "
                "2020→21 also spans COVID.")
layout(fig, top=0.86, bottom=0.08)
save(fig, "07_ces_common_vs_partisan")

## 7. Midterm placebo: does opinion follow Congress too?
**Logic:** if economic ratings track *power in general*, the party that wins the House should gain on the party that
lost it. If they track *the presidency*, midterms should barely move the gap. The estimand is the change in the
**winning-minus-losing gap**, because in 2022 every group rose 8–11 points together (a common shock that moves levels but
not the gap).

**Read the two panels differently.** UMich (left) is the primary placebo: bars are its own noise band. CES (right) is
supporting context only: its annual gap drifts up to ~17 points with no election at all (section 6b), so a CES midterm
move has to clear that dashed line, not the tiny sampling band.

In [0]:
um = placebo[placebo["source"] == "UMich (monthly)"].copy()
cs = placebo[placebo["source"] == "CES (annual)"].copy()
ORDER = ["mid-2018", "mid-2022", "pres-2020", "pres-2024"]
NAMES = {"mid-2018": "2018 midterm (House → Dem.)", "mid-2022": "2022 midterm (House → Rep.)",
         "pres-2020": "2020 presidential", "pres-2024": "2024 presidential", "pres-2016": "2016 presidential"}

fig, (axL, axR) = plt.subplots(1, 2, figsize=(12.5, 4.6), gridspec_kw={"width_ratios": [1.2, 1]})
for i, ev in enumerate(ORDER[::-1]):
    for cp, mk, off in [(3, "o", 0.17), (8, "D", -0.17)]:
        r = um[(um["event_key"] == ev) & (um["checkpoint_months_from_election"] == cp)]
        if r.empty:
            continue
        r = r.iloc[0]
        col = INK if not r["is_placebo"] else "#8C8C8C"
        axL.errorbar(r["winner_minus_loser_gap_change"], i + off, xerr=r["gap_change_noise_band"], fmt=mk, color=col,
                     ms=7, mec="white", capsize=0, lw=1.6)
        # Value label to the right of the noise-band whisker, so it never sits on the marker or the bar
        axL.text(r["winner_minus_loser_gap_change"] + r["gap_change_noise_band"] + 1.5, i + off,
                 f"{r['winner_minus_loser_gap_change']:+.0f}", ha="left", va="center", fontsize=7.5, color=INK)
axL.set_yticks(range(len(ORDER)), [NAMES[e] for e in ORDER[::-1]])
axL.axvline(0, color=MUTED, lw=0.8)
axL.set_xlabel("Change in winning-minus-losing party gap (UMich index points)")
axL.set_title("UMich monthly (primary placebo)", fontsize=10.5)
axL.grid(axis="x", color=GRID); axL.grid(axis="y", visible=False)
axL.legend(handles=[Line2D([], [], marker="o", ls="", color=INK, label="3 months after election"),
                    Line2D([], [], marker="D", ls="", color=INK, label="6 months after takeover"),
                    Line2D([], [], color=INK, lw=1.6, label="± noise band (2 SE)")], loc="lower left", fontsize=8)
axL.set_xlim(-15, um["winner_minus_loser_gap_change"].max() + um["gap_change_noise_band"].max() + 10)

cs = cs.sort_values(["is_placebo", "event_key"])
for i, (_, r) in enumerate(cs.iterrows()):
    col = INK if not r["is_placebo"] else "#8C8C8C"
    axR.plot(r["winner_minus_loser_gap_change"], i, "o", ms=8, color=col, mec="white")
    axR.text(r["winner_minus_loser_gap_change"], i + 0.22, f"{r['winner_minus_loser_gap_change']:+.0f}", ha="center",
             fontsize=7.5, color=INK)
qmax = cs["quiet_years_max_abs_gap_change"].dropna().iloc[0]
axR.axvspan(-qmax, qmax, color=BAND, alpha=0.6, lw=0, zorder=0)
axR.text(0, len(cs) - 0.4, f"±{qmax:.0f}: largest gap drift with no election", ha="center", fontsize=7.5, color=MUTED)
axR.set_yticks(range(len(cs)), [NAMES.get(e, e) + f"\n{lab.split('(')[1].rstrip(')')}" for e, lab in zip(cs["event_key"], cs["checkpoint_label"])],
               fontsize=8)
axR.axvline(0, color=MUTED, lw=0.8)
axR.set_xlabel("Change in winning-minus-losing gap (pts, % better)")
axR.set_title("CES annual (context only)", fontsize=10.5)
axR.grid(axis="x", color=GRID); axR.grid(axis="y", visible=False)
axR.set_ylim(-0.6, len(cs) - 0.1)

add_titles(fig, "Winning the White House moves the partisan gap. Winning the House does not.",
           "Change in the gap between the party that won and the party that lost, after midterms (gray) and presidential "
           "elections (black). Every midterm here was won by the party out of the White House.")
add_source(fig, f"Sources: {SOURCE_UMICH}; {SOURCE_CES}. fct_midterm_placebo. CES midterm pairs: 2018→19 is driven by Republicans "
                "falling while their party held the White House; 2022→23 the losing party gained.")
layout(fig, top=0.84, bottom=0.09)
save(fig, "08_midterm_placebo")

## 8. The co-partisan premium (RQ1 model)
**The model:** a weighted linear probability model on 383,533 CES respondents with **year fixed effects** (absorbing
everything national in a year, including the economy itself), party, demographic controls and state fixed effects. The
co-partisan premium β₃ is identified only from changes in who holds the White House.

**The forest plot is the robustness battery for headline number 1.** Every pre-specified spec from notebook 04 is shown
with its state-clustered 95% interval. The mirror outcome ("worse") is plotted with its **sign flipped**, and labeled so,
so that all rows point the same way; the 5-point score is in different units and is listed under the chart instead.

In [0]:
FOREST = [("main", "co_partisan", "Main specification", 1),
          ("split_by_party", "co_partisan_rep", "  Republicans only (split model)", 1),
          ("split_by_party", "co_partisan_dem", "  Democrats only (split model)", 1),
          ("ns_as_same", "co_partisan", "'Not sure' counted as 'same'", 1),
          ("leaners_as_independents", "co_partisan", "Leaners as independents", 1),
          ("no_controls", "co_partisan", "No controls, no state FE", 1),
          ("excluding_covid", "co_partisan", "Excluding 2020–21", 1),
          ("year_specific_weights", "co_partisan", "Year-specific weights", 1),
          ("unweighted", "co_partisan", "Unweighted", 1),
          ("logit_ame", "co_partisan", "Logit, average marginal effect", 1),
          ("outcome_worse", "co_partisan", "Mirror outcome: % worse (sign flipped)", -1)]

fig, ax = plt.subplots(figsize=(10.5, 5.8))
main_est = one(rq1, "estimate", spec="main", term="co_partisan")
for i, (spec, term, lab, sign) in enumerate(FOREST[::-1]):
    r = rq1[(rq1["spec"] == spec) & (rq1["term"] == term)].iloc[0]
    est, lo, hi = sign * r["estimate"], sign * r["ci_low_cluster"], sign * r["ci_high_cluster"]
    lo, hi = min(lo, hi), max(lo, hi)
    color = COLORS["Republican"] if term.endswith("_rep") else COLORS["Democrat"] if term.endswith("_dem") else INK
    ax.plot([lo, hi], [i, i], color=color, lw=2.2)
    ax.plot(est, i, "s" if spec == "main" else "o", ms=8 if spec == "main" else 6.5, color=color, mec="white")
    ax.text(hi + 0.25, i, f"{est:.1f}  ({lo:.1f}–{hi:.1f})", va="center", fontsize=8, color=INK)
ax.axvline(main_est, color=MUTED, ls=":", lw=1)
ax.set_yticks(range(len(FOREST)), [f[2] for f in FOREST[::-1]])
ax.set_xlim(30, 50)
ax.set_xlabel("Co-partisan premium (percentage points more likely to say the economy got better)")
ax.grid(axis="x", color=GRID); ax.grid(axis="y", visible=False)
s5 = rq1[(rq1["spec"] == "outcome_score5") & (rq1["term"] == "co_partisan")].iloc[0]
add_titles(fig, "Having your party in the White House adds about 40 points, whichever party it is",
           "Difference in the probability of saying the national economy got better, for a supporter of the president's "
           "party vs the same person when their party is out, holding the year (and so the economy) fixed.")
add_source(fig, f"Source: {SOURCE_CES}, 2016–2025; fct_rq1_model_estimates. WLS with year and state fixed effects and demographic "
                f"controls; state-clustered 95% intervals. 5-point score spec (not shown): +{s5['estimate']:.2f} scale points "
                f"({s5['ci_low_cluster']:.2f}–{s5['ci_high_cluster']:.2f}). Logit interval is approximate.")
layout(fig, top=0.85, bottom=0.10)
save(fig, "09_rq1_premium_forest")

### 8a. Year effects: the common, non-partisan component
The year fixed effects α_t are what is left once party and the White House are accounted for: the part of "the economy
got better" that moved for everyone. They feed RQ2 (does that common part track fundamentals?). Note 2017 (+21) and 2022
(−16, the inflation year): the common component moves in the directions the economy did.

In [0]:
ye = M["fct_rq1_year_effects"].sort_values("survey_year")
fig, ax = plt.subplots(figsize=(10, 4.2))
ax.bar(ye["survey_year"], ye["year_effect_pp"], color=COLORS["All respondents"], width=0.62)
ax.errorbar(ye["survey_year"], ye["year_effect_pp"],
            yerr=[ye["year_effect_pp"] - ye["ci_low_cluster"], ye["ci_high_cluster"] - ye["year_effect_pp"]],
            fmt="none", ecolor=MUTED, lw=1.2)
for _, r in ye.iterrows():
    v = r["year_effect_pp"]
    # Label beyond the end of the interval, not the bar, so the whisker and the number never overlap
    y_lab = r["ci_high_cluster"] + 0.8 if v >= 0 else r["ci_low_cluster"] - 0.8
    ax.text(r["survey_year"], y_lab, f"{v:+.0f}" if r["survey_year"] != 2016 else "ref.", ha="center",
            va="bottom" if v >= 0 else "top", fontsize=8.5, color=INK)
ax.set_ylim(ye["ci_low_cluster"].min() - 4, ye["ci_high_cluster"].max() + 4)
ax.axhline(0, color=MUTED, lw=0.8)
ax.set_xticks(ye["survey_year"])
ax.set_ylabel("Year effect vs 2016 (pp)")
add_titles(fig, "The part of economic perception everyone shares rose in 2017 and fell in 2022",
           "Year fixed effects from the main RQ1 model: the change in '% better' common to all respondents, relative to 2016.")
add_source(fig, f"Source: {SOURCE_CES}; fct_rq1_year_effects. State-clustered 95% intervals.")
layout(fig, top=0.85, bottom=0.08)
save(fig, "10_rq1_year_effects")

## 9. Your wallet vs "the economy" (RQ3)
For each ANES year, the **partisan gap** = % saying "worse" among the party out of the White House minus the party in it,
asked about the national economy and about the respondent's own finances. If partisanship colored everything equally the
bars would be the same height. Intervals are the respondent bootstrap from notebook 04 (2,000 draws; no strata/PSU in
the cumulative file, so they are somewhat too narrow, a listed gap).

In [0]:
b = pb_boot[pb_boot["is_main_spec"]].sort_values("survey_year")
in_party = pb.set_index("survey_year")["white_house_party"].groupby(level=0).first()
fig, ax = plt.subplots(figsize=(10.5, 5))
x = np.arange(len(b))
for off, col, lab, key in [(-0.19, COLORS["All respondents"], "National economy", "national"),
                           (0.19, "#A8A8A8", "Own finances", "personal")]:
    vals = b[f"{key}_gap_pts"].to_numpy()
    ax.bar(x + off, vals, width=0.36, color=col, label=lab)
    ax.errorbar(x + off, vals, yerr=[vals - b[f"{key}_gap_ci_low"], b[f"{key}_gap_ci_high"] - vals], fmt="none",
                ecolor=INK, lw=1.1)
    for xi, v in zip(x + off, vals):
        ax.text(xi, 1.2, f"{v:.0f}", ha="center", va="bottom", fontsize=9, color="white" if key == "national" else INK,
                fontweight="bold")
for xi, (_, r) in zip(x, b.iterrows()):
    ax.text(xi, r["national_gap_ci_high"] + 2.5,
            f"{r['national_to_personal_ratio']:.1f}× ({r['national_to_personal_ratio_ci_low']:.1f}–{r['national_to_personal_ratio_ci_high']:.1f})",
            ha="center", fontsize=9, color=INK, fontweight="bold")
ax.set_xticks(x, [f"{int(y)}\n({in_party[y][:3]}. White House)" for y in b["survey_year"]])
ax.set_ylabel("Out-party minus in-party, % saying 'worse' (pts)")
ax.set_ylim(0, b["national_gap_ci_high"].max() + 10)
ax.legend(loc="upper left", fontsize=9)
add_titles(fig, "People's view of their own finances is two to three and a half times less partisan",
           "Partisan gap on the national economy vs on respondents' own finances, with the national ÷ personal ratio above each pair.")
add_source(fig, f"Source: {SOURCE_ANES}; fct_pocketbook_gap_bootstrap (main spec: 'do not know' excluded, leaners with parties). "
                "95% respondent-bootstrap intervals; no design variables in the cumulative file, so intervals are a lower bound.")
layout(fig, top=0.86, bottom=0.09)
save(fig, "11_rq3_wallet_vs_economy")

### 9a. Does the difference survive controls? (the "income realignment" objection)
If out-partisans were genuinely worse off (say, because the parties' income mix changed), their pocketbook answers
should be gloomier for real and the national-vs-personal difference could be a composition artifact. The stacked model
from notebook 04 answers this: **b₃** is how much more partisan "the economy" is than the wallet, with and without
income, employment, homeownership and demographic controls. It does not move.

In [0]:
ctrl = (pb_model.pivot_table(index=["survey_year", "term"], columns="with_controls", values=["estimate_pp", "ci_low", "ci_high"])
        .sort_index())
ctrl.columns = [f"{v}_{'controls' if c else 'no_controls'}" for v, c in ctrl.columns]
ctrl = ctrl.reset_index()
ctrl["shift_from_controls_pp"] = ctrl["estimate_pp_controls"] - ctrl["estimate_pp_no_controls"]
display(ctrl[["survey_year", "term", "estimate_pp_no_controls", "estimate_pp_controls", "ci_low_controls",
              "ci_high_controls", "shift_from_controls_pp"]]
        .style.format(precision=1).hide(axis="index")
        .set_caption("RQ3 stacked LPM: gaps in pp, respondent-clustered 95% CI (controls: income tercile, employment, "
                     "homeownership, age, education, race, sex)"))
if SAVE_OUTPUTS:
    ctrl.to_csv(TABLE_DIR / "rq3_controls_model.csv", index=False)

## 10. Robustness battery for the two headline numbers
The plan promises a robustness battery "for every headline number". The two headline numbers are:
1. **The partisan flip** (RQ1): the winning-minus-losing gap moves by tens of points at a change of party, ~40 pp in the model.
2. **The wallet is less partisan** (RQ3): the national-economy gap exceeds the own-finances gap in every year.

Each row is one alternative choice, with its estimate, interval (where one exists), and whether it points the **same
direction** as the main result. A battery item that flips the direction would be the first thing to say to the panel;
none does. The table is exported as `robustness_battery.csv` for the methods box.

In [0]:
rob = []

# ── Headline 1a: model specifications (pp of "better" unless noted) ─────────
for spec, term, lab, sign in FOREST + [("outcome_score5", "co_partisan", "5-point score (scale points)", 1)]:
    r = rq1[(rq1["spec"] == spec) & (rq1["term"] == term)].iloc[0]
    rob.append(dict(headline="1 · Partisan flip", source="CES model", event="Pooled 2016–2025", check=lab.strip(),
                    estimate=r["estimate"], ci_low=r["ci_low_cluster"], ci_high=r["ci_high_cluster"], unit=r["unit"],
                    expected_sign=sign))

# ── Headline 1b: CES wave contrasts at each change of party, every measure × party definition ──────────
MEASURE_SIGN = {"pct_better": 1, "pct_better_ns_as_same": 1, "net_rating": 1, "mean_score5": 1, "pct_better_unweighted": 1,
                "pct_worse": -1}
ces_t = contrasts[(contrasts["source"] == "CES") & (contrasts["comparison_type"] == "Presidential transition")]
for _, r in ces_t.iterrows():
    se = r["se_winner_minus_loser_gap_change"]
    rob.append(dict(headline="1 · Partisan flip", source="CES contrast", event=r["event_label"],
                    check=f"{r['measure_label']} · {r['party_definition']}", estimate=r["winner_minus_loser_gap_change"],
                    ci_low=r["winner_minus_loser_gap_change"] - 1.96 * se if pd.notna(se) else np.nan,
                    ci_high=r["winner_minus_loser_gap_change"] + 1.96 * se if pd.notna(se) else np.nan,
                    unit=r["measure_unit"], expected_sign=MEASURE_SIGN[r["measure"]]))

# ── Headline 1c: other surveys (Pew both windows; UMich at six months after takeover) ─────────
for _, r in contrasts[contrasts["source"] == "Pew"].iterrows():
    rob.append(dict(headline="1 · Partisan flip", source="Pew", event=r["event_label"], check=r["comparison_window"],
                    estimate=r["winner_minus_loser_gap_change"], ci_low=np.nan, ci_high=np.nan, unit="percentage points",
                    expected_sign=1))
for _, r in placebo[(placebo["source"] == "UMich (monthly)") & ~placebo["is_placebo"].astype(bool)
                    & (placebo["checkpoint_months_from_election"] == 8)].iterrows():
    rob.append(dict(headline="1 · Partisan flip", source="UMich", event=r["event_label"], check="Gap change, six months after takeover",
                    estimate=r["winner_minus_loser_gap_change"], ci_low=r["winner_minus_loser_gap_change"] - r["gap_change_noise_band"],
                    ci_high=r["winner_minus_loser_gap_change"] + r["gap_change_noise_band"], unit="index points", expected_sign=1))

# ── Headline 2: RQ3 across outcome spec × party definition, plus the controlled model ─────────
for _, r in pb_boot.iterrows():
    rob.append(dict(headline="2 · Wallet less partisan", source="ANES", event=str(r["survey_year"]),
                    check=f"National − personal gap · {r['outcome_spec']} · {r['party_definition']}",
                    estimate=r["gap_difference_pts"], ci_low=r["gap_difference_ci_low"], ci_high=r["gap_difference_ci_high"],
                    unit="percentage points", expected_sign=1))
for _, r in pbm_b3.iterrows():
    rob.append(dict(headline="2 · Wallet less partisan", source="ANES model", event=str(r["survey_year"]),
                    check=f"Stacked LPM b3 · {'with' if r['with_controls'] else 'without'} controls",
                    estimate=r["estimate_pp"], ci_low=r["ci_low"], ci_high=r["ci_high"], unit="percentage points", expected_sign=1))

robustness = pd.DataFrame(rob)
robustness["same_direction"] = np.sign(robustness["estimate"]) == robustness["expected_sign"]
robustness["interval_excludes_zero"] = np.where(robustness["ci_low"].isna(), np.nan,
                                                (robustness["ci_low"] > 0) | (robustness["ci_high"] < 0))

# Group by unit too: net ratings (−100..100) and 5-point means are not on the same scale as shares
summary = (robustness.groupby(["headline", "source", "unit"])
           .agg(checks=("estimate", "size"), same_direction=("same_direction", "sum"),
                min_estimate=("estimate", "min"), max_estimate=("estimate", "max"))
           .reset_index())
display(summary.style.format(precision=1).hide(axis="index").set_caption("Robustness battery, summary"))
display(robustness.style.format(precision=1, na_rep="—").hide(axis="index").set_caption("Robustness battery, every check"))
if SAVE_OUTPUTS:
    robustness.to_csv(TABLE_DIR / "robustness_battery.csv", index=False)

### 10a. The plan's robustness checklist, honestly scored
The plan lists seven battery items. This cell scores each against what the marts actually contain, so the panel hears
what was **not** done as clearly as what was. (Ordered logit and county controls were planned and not built.)

In [0]:
has = lambda spec: spec in set(rq1["spec"])                                                        # noqa: E731
checklist = pd.DataFrame([
    ("Weighted vs unweighted", has("unweighted") and "pct_better_unweighted" in set(contrasts["measure"]),
     "Model `unweighted` spec; CES contrast measure `pct_better_unweighted`"),
    ("Binary vs 5-point mean vs ordered logit", has("outcome_score5"),
     "Binary and 5-point done (`outcome_score5`, `mean_score5`). **Ordered logit not built** (logit AME is the binary check)"),
    ("'Not sure' excluded vs folded into 'same'", has("ns_as_same"), "Model `ns_as_same`; contrasts; RQ3 'do not know in base'"),
    ("Leaners with parties vs as independents", has("leaners_as_independents"), "Model, contrasts and RQ3 carry both definitions"),
    ("With and without demographic and county controls", has("no_controls"),
     "Demographic + state FE done (`no_controls`; RQ3 b3 with/without). **County unemployment controls not built** (LAUS is a data gap)"),
    ("Excluding 2020–2021 (COVID)", has("excluding_covid"), "Model `excluding_covid`; `is_covid_comparison` flag; RQ4 sample"),
    ("Same direction in all three surveys", robustness.query("headline == '1 · Partisan flip' and source in ['CES contrast','Pew','UMich']")["same_direction"].all(),
     "CES, Pew and UMich all show the winner gaining on the loser at every change of party (section 13d)"),
], columns=["battery item", "done", "evidence / gap"])
display(checklist.style.hide(axis="index"))

## 11. RQ2: is anything left of the fundamentals?
### 11a. Expected vs actual inflation
UMich's median year-ahead price expectation against actual headline inflation (past 12 months), monthly. The median is
the primary measure because a few extreme answers pull the mean above actual inflation in most months. The shaded area is
the gap: above the actual line, people expect prices to run hotter than the news says they just did.

In [0]:
ev_ = M["fct_expected_vs_actual_inflation"].sort_values("month")
fig, (ax, ax2) = plt.subplots(2, 1, figsize=(12, 7), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
ax.plot(ev_["month"], ev_["actual_inflation_current_pct"], color=COLORS["All respondents"], lw=2, label="Actual inflation (CPI, y/y)")
ax.plot(ev_["month"], ev_["expected_inflation_1y_median_pct"], color="#8A6BBE", lw=2, label="Expected, next 12 months (median)")
ax.fill_between(ev_["month"], ev_["actual_inflation_current_pct"], ev_["expected_inflation_1y_median_pct"],
                where=ev_["expected_inflation_1y_median_pct"] > ev_["actual_inflation_current_pct"], color="#8A6BBE", alpha=0.15, lw=0)
peak = ev_.loc[ev_["expected_inflation_1y_median_pct"].idxmax()]
ax.annotate(f"Peak expectation {peak['expected_inflation_1y_median_pct']:.1f}%\n({peak['month']:%b %Y})",
            (peak["month"], peak["expected_inflation_1y_median_pct"]), xytext=(-110, 5), textcoords="offset points",
            fontsize=8.5, color=INK, arrowprops=dict(arrowstyle="-", color=MUTED))
gap_m = ev_[ev_["is_shutdown_gap_month"].astype(bool)]
for m in gap_m["month"]:
    ax.axvline(m, color=MUTED, ls=":", lw=1)
    ax.text(m, ax.get_ylim()[1] * 0.02 + 0.3, " no CPI\n (shutdown)", fontsize=7, color=MUTED)
ax.set_ylabel("Percent")
ax.legend(loc="upper left", fontsize=8.5)
ax.set_title("Median expected inflation vs actual inflation", fontsize=10.5)

ax2.bar(ev_["month"], ev_["expected_minus_current_inflation_pct"], width=25,
        color=np.where(ev_["expected_minus_current_inflation_pct"] > 0, "#8A6BBE", "#A8A8A8"))
ax2.axhline(0, color=MUTED, lw=0.8)
ax2.set_ylabel("Expected − actual (pts)")
ax2.set_title("The gap, month by month", fontsize=10.5)
ax2.xaxis.set_major_locator(mdates.YearLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
add_titles(fig, "Expectations missed low in 2022 and high in 2025",
           f"Median expected inflation ran {abs(N['exp_2022']):.1f} points below actual in 2022 and {N['exp_2025']:.1f} above "
           "in 2025. Whether the 2025 gap is partisan needs expectations by party, which the repo does not have.")
add_source(fig, "Sources: UMich Surveys of Consumers (px1_med, all respondents); monthly CPI inflation. October 2025 CPI was not "
                "published and is left blank, not interpolated. Purple is used for expectations so it never reads as a party.")
layout(fig, top=0.86, bottom=0.08)
save(fig, "12_rq2_expected_vs_actual_inflation")

### 11b · Which prices do expectations track? (the salience test)
Average absolute gap between the median expectation and each price series, by year. A smaller bar = expectations sit
closer to that series. The plan's hypothesis is that *visible* prices (gas, groceries) explain expectations better than
core inflation. CPI component year-over-year changes start in 2017.

In [0]:
sal = evi[evi["year"].notna() & (evi["year"] >= 2017)].copy()
sal["year"] = sal["year"].astype(int)
sal = sal.sort_values("year")        # GROUPING SETS output has no guaranteed order; sort before plotting
SERIES = [("avg_abs_gap_to_gasoline_pct", "Gasoline", "#D9A441"), ("avg_abs_gap_to_food_at_home_pct", "Food at home", "#6E9E5E"),
          ("avg_abs_gap_to_core_pct", "Core CPI", "#7F7F7F"), ("avg_abs_gap_to_headline_pct", "Headline CPI", INK)]
fig, ax = plt.subplots(figsize=(12, 4.6))
w = 0.2
for j, (col, lab, c) in enumerate(SERIES):
    ax.bar(np.arange(len(sal)) + (j - 1.5) * w, sal[col], width=w - 0.02, color=c, label=lab)
ax.set_xticks(np.arange(len(sal)), sal["year"])
ax.set_ylabel("Avg |expected − actual| (pts)")
ax.set_ylim(0, sal[[c for c, _, _ in SERIES]].max().max() * 1.15)   # headroom so the legend clears the tallest bar
ax.legend(ncol=4, loc="upper left", fontsize=8.5)
add_titles(fig, "Price expectations stay close to overall inflation, nowhere near the swings in gas prices",
           "Average absolute gap between median expected inflation and each price series' year-over-year change. Lower = closer. "
           "This level comparison cannot say which prices people react to; the change-based test is in 11c.")
add_source(fig, "Sources: UMich px1_med; BLS CPI components (SA); fct_expected_vs_actual_inflation_by_year. Gas prices swing far more "
                "than expectations do, so large gas gaps partly reflect gas volatility; descriptive only. 2026 is a partial year.")
layout(fig, top=0.86, bottom=0.09)
save(fig, "13_rq2_salience_gaps")

### 11c. Does the non-partisan component track fundamentals? (correlation grid)
Correlations between **year-over-year changes** in the two non-partisan perception measures (independents; RQ1 year
effects) and changes in each fundamental, primary 3-month window. Cells marked ● have a bootstrap interval excluding
zero. With 8–9 year-pairs per cell and 63 cells in the full grid, about three would cross zero by chance, so this is
**descriptive**: the only confirmatory contrast is gas vs core (printed below the chart), and it is null. The consistent
exploratory lead is **food at home** (negative, as expected if grocery inflation sours perceptions).

In [0]:
rq2 = M["fct_rq2_correlations"]
grid = rq2[rq2["is_primary_window"].astype(bool) & ~rq2["fundamental"].str.startswith("SALIENCE")]
mat = grid.pivot(index="perception_measure", columns="fundamental", values="r")
sig = grid.assign(excl=(grid["ci_low"] > 0) | (grid["ci_high"] < 0)).pivot(index="perception_measure", columns="fundamental", values="excl")
n_pairs = grid.pivot(index="perception_measure", columns="fundamental", values="n_year_pairs")
col_order = ["Headline inflation", "Core CPI", "Gasoline CPI", "Food-at-home CPI", "Shelter CPI", "Unemployment", "Real hourly earnings"]
mat, sig, n_pairs = mat[col_order], sig[col_order], n_pairs[col_order]

fig, ax = plt.subplots(figsize=(11.5, 3.8))
im = ax.imshow(mat.to_numpy(float), cmap="PuOr_r", vmin=-1, vmax=1, aspect="auto")   # purple–orange: diverging, no party hues
for i in range(mat.shape[0]):
    for j in range(mat.shape[1]):
        v = mat.iat[i, j]
        if pd.notna(v):
            ax.text(j, i, f"{v:+.2f}{' ●' if sig.iat[i, j] else ''}\n(n={int(n_pairs.iat[i, j])})", ha="center", va="center",
                    fontsize=8, color="white" if abs(v) > 0.6 else INK)
ax.set_xticks(range(len(col_order)), col_order, rotation=0, fontsize=8.5)
ax.set_yticks(range(len(mat.index)), mat.index, fontsize=9)
ax.grid(False)
cb = fig.colorbar(im, ax=ax, fraction=0.025, pad=0.01)
cb.set_label("Pearson r (changes)", fontsize=8)
sc = rq2[rq2["fundamental"].str.startswith("SALIENCE")].iloc[0]
add_titles(fig, "One consistent lead: non-partisan perceptions sour when grocery inflation rises",
           f"Correlation of year-over-year changes, 3-month window before fieldwork. ● = 95% bootstrap interval excludes zero. "
           f"Pre-specified gas-vs-core contrast: {sc['r']:+.2f} ({sc['ci_low']:+.2f} to {sc['ci_high']:+.2f}), no evidence gas matters more.")
add_source(fig, f"Sources: {SOURCE_CES}; fct_rq1_year_effects; BLS CPI; fct_rq2_correlations. Exploratory: cells share one food series, "
                "overlapping windows and correlated measures, so they are not independent tests (see the Holm adjustment in section 13c).")
layout(fig, top=0.80, bottom=0.10)
save(fig, "14_rq2_correlation_grid")

## 12. RQ4: say vs do (context for the Imprint extension)
Aggregate spending cannot separate Republicans from Democrats, so this is context, not a test. The question it sets up
is the one only card data can answer: **do partisan swings in what people say show up in what they spend?** In the
aggregate, party sentiment swung 30–50 points while real spending grew steadily (top vs bottom panel), and monthly changes
in party sentiment are essentially uncorrelated with spending growth (forest plot).

In [0]:
sd = M["fct_say_vs_do_monthly"].sort_values("month")
sd = sd[sd["month"] >= "2017-04-01"]
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6.6), sharex=True, gridspec_kw={"height_ratios": [1.6, 1]})
for s, col in [("Republican", "sentiment_rep"), ("Democrat", "sentiment_dem"), ("Independent", "sentiment_ind")]:
    ax1.plot(sd["month"], sd[col], color=COLORS[s], lw=1.8, label=s)
    direct_label(ax1, sd["month"].iloc[-1], sd[col].iloc[-1], s, COLORS[s], dx=6)
ax1.set_ylabel("Sentiment index")
ax1.set_title("What people say: UMich sentiment by party", fontsize=10.5)
ax2.bar(sd["month"], sd["pce_real_mom_pct"], width=25, color=np.where(sd["pce_real_mom_pct"] >= 0, COLORS["All respondents"], "#A8A8A8"))
ax2.axhline(0, color=MUTED, lw=0.8)
ax2.set_ylim(-3, 3)    # clip the April 2020 collapse (−11%) so normal months are readable; noted in the source line
ax2.set_ylabel("Real PCE, % m/m")
ax2.set_title("What households do: real consumer spending growth (aggregate)", fontsize=10.5)
ax2.xaxis.set_major_locator(mdates.YearLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax2.set_xlim(sd["month"].min(), sd["month"].max() + pd.DateOffset(months=8))
add_titles(fig, "Sentiment swings with elections; spending mostly does not",
           "Party sentiment (top) moved 30–50 points around each change of party; month-to-month real spending growth (bottom) "
           "shows no matching break outside the 2020 pandemic shock.")
add_source(fig, f"Sources: {SOURCE_UMICH}; BEA real personal consumption expenditures (PCEC96, revised). Bottom axis clipped at ±3%: "
                "March–April 2020 fell about 7% and 11%, and the spring 2020 rebound and March 2021 exceed +3%. "
                "Aggregate data cannot show whether partisans spend differently.")
layout(fig, top=0.86, bottom=0.08)
save(fig, "15_rq4_say_vs_do")

In [0]:
c = sdc[sdc["do_measure"] == "Real PCE growth (% m/m)"].copy()
SAY_COLOR = {"Republican sentiment": COLORS["Republican"], "Democratic sentiment": COLORS["Democrat"],
             "Independent sentiment": COLORS["Independent"], "All respondents' sentiment": COLORS["All respondents"]}
fig, ax = plt.subplots(figsize=(10, 4.3))
rows = [(s, smp) for s in SAY_COLOR for smp in ["All months", "Excluding 2020-2021 (plan robustness)"]]
for i, (s, smp) in enumerate(rows[::-1]):
    r = c[(c["say_measure"] == s) & (c["sample"] == smp)].iloc[0]
    ax.plot([r["ci_low"], r["ci_high"]], [i, i], color=SAY_COLOR[s], lw=2)
    ax.plot(r["r"], i, "o" if smp == "All months" else "D", color=SAY_COLOR[s], ms=7, mec="white")
    ax.text(r["ci_high"] + 0.02, i, f"{r['r']:+.2f}", va="center", fontsize=8, color=INK)
SHORT = {"Republican sentiment": "Republicans", "Democratic sentiment": "Democrats",
         "Independent sentiment": "Independents", "All respondents' sentiment": "All respondents"}
ax.set_yticks(range(len(rows)), [f"{SHORT[s]} · {'all months' if smp == 'All months' else 'excl. 2020–21'}"
                                 for s, smp in rows[::-1]], fontsize=8.5)
ax.axvline(0, color=MUTED, lw=0.8)
ax.set_xlim(-0.6, 0.8)
ax.set_xlabel("Correlation of monthly sentiment change with real PCE growth (95% moving-block bootstrap)")
ax.grid(axis="x", color=GRID); ax.grid(axis="y", visible=False)
add_titles(fig, "The only sentiment–spending link is the 2020 pandemic collapse",
           "Party sentiment changes are uncorrelated with spending growth in either sample; the all-respondents correlation "
           "vanishes once 2020–21 is excluded.")
add_source(fig, "Source: fct_say_vs_do_correlations. Block bootstrap (6-month blocks) for autocorrelation. Lead-lag correlations "
                "deliberately not computed until lags are pre-specified.")
layout(fig, top=0.85, bottom=0.10)
save(fig, "16_rq4_correlations")

## 13. Further analysis (computed here, from marts only)
These answer likely panel questions that the marts make cheap to check. None uses microdata, none changes a headline
number, and each is labeled with its status.

### 13a. Is the premium really symmetric? *(robustness)*
The split model gives Republicans 38.2 and Democrats 41.8. Is a 3.6-point difference meaningful? The test below uses the
two clustered SEs and treats the estimates as independent, which ignores their covariance in the joint fit, so the
interval is approximate. **Interpretation rule set in advance:** "roughly symmetric" if the difference is small relative
to the premium itself, whether or not its interval excludes zero.

In [0]:
rep_r = rq1[(rq1["spec"] == "split_by_party") & (rq1["term"] == "co_partisan_rep")].iloc[0]
dem_r = rq1[(rq1["spec"] == "split_by_party") & (rq1["term"] == "co_partisan_dem")].iloc[0]
diff = dem_r["estimate"] - rep_r["estimate"]
se_diff = np.hypot(rep_r["se_cluster_state"], dem_r["se_cluster_state"])
asym = pd.DataFrame([{
    "Republican premium (pp)": rep_r["estimate"], "Democratic premium (pp)": dem_r["estimate"],
    "Dem − Rep (pp)": diff, "approx. 95% CI low": diff - 1.96 * se_diff, "approx. 95% CI high": diff + 1.96 * se_diff,
    "difference as % of main premium": 100 * diff / main_est,
}])
display(asym.style.format(precision=1).hide(axis="index"))
print(f"Asymmetry is {abs(diff):.1f} pp, {100 * abs(diff) / main_est:.0f}% of the premium: "
      f"{'roughly symmetric' if abs(diff) / main_est < 0.15 else 'worth reporting as asymmetric'} by the pre-set rule.")

### 13b. Does the timing result depend on the baseline or the endpoint? *(robustness)*
The swing-timing estimand uses a July–September baseline and an endpoint six months after the inauguration (+8 months
from the election). Both are defensible choices, so the obvious question is whether others would tell a different story.
This recomputes the share done at the election and the inauguration for three baselines × three endpoints, straight from
the monthly levels in `fct_event_study`. **The finding to check:** the move is partly done by the election, partly by the
inauguration, and incomplete at the inauguration, under every combination.

In [0]:
BASELINES = {"Jun–Aug (−5..−3)": [-5, -4, -3], "Jul–Sep (−4..−2) · main": [-4, -3, -2], "Aug–Oct (−3..−1)": [-3, -2, -1]}
ENDPOINTS = {"+6 (May)": 6, "+8 (July) · main": 8, "+10 (Sep)": 10}
sens = []
for ev, _ in EVENTS_UMICH:
    for s in ["Republican", "Democrat"]:
        lv = es[(es["event_key"] == ev) & (es["series"] == s)].set_index("months_from_election")["sentiment_index"]
        for bl, bmonths in BASELINES.items():
            base = lv.reindex(bmonths).mean()
            for ep, e in ENDPOINTS.items():
                total = lv.get(e, np.nan) - base
                sens.append(dict(event=LABEL_EVENT[ev], series=s, baseline=bl, endpoint=ep,
                                 pct_done_at_election=100 * (lv.get(0, np.nan) - base) / total,
                                 pct_done_at_inauguration=100 * (lv.get(2, np.nan) - base) / total))
sens = pd.DataFrame(sens)
sens_wide = sens.pivot_table(index=["event", "series", "baseline"], columns="endpoint",
                             values=["pct_done_at_election", "pct_done_at_inauguration"]).round(0)
display(sens_wide)
rng_ = sens.groupby(["event", "series"]).agg(election_min=("pct_done_at_election", "min"), election_max=("pct_done_at_election", "max"),
                                             inaug_min=("pct_done_at_inauguration", "min"), inaug_max=("pct_done_at_inauguration", "max")).round(0)
display(rng_)
print("Share done at inauguration stays below 100% under every choice:", bool((sens["pct_done_at_inauguration"] < 100).all()))
if SAVE_OUTPUTS:
    sens.to_csv(TABLE_DIR / "swing_timing_sensitivity.csv", index=False)

### 13c. Multiple comparisons in the RQ2 grid *(exploratory → adjusted)*
The plan says "everything else labeled exploratory, with Holm adjustment where tests are reported". The bootstrap intervals
in section 11c are the reported intervals; this adds the matching p-values (t-test on r with n − 2 df, which is
approximate for 8–9 points) and a Holm correction across all 63 cells. If nothing survives Holm, the food-at-home pattern
stays a lead for a follow-up story, not a finding, which is exactly how the plan frames it.

In [0]:
g63 = rq2[~rq2["fundamental"].str.startswith("SALIENCE")].copy()
t_stat = g63["r"] * np.sqrt((g63["n_year_pairs"] - 2) / (1 - g63["r"] ** 2))
g63["p_raw"] = 2 * stats.t.sf(np.abs(t_stat), df=g63["n_year_pairs"] - 2)
g63["p_holm"] = multipletests(g63["p_raw"], method="holm")[1]
g63["boot_ci_excludes_zero"] = (g63["ci_low"] > 0) | (g63["ci_high"] < 0)
print(f"Cells: {len(g63)} · raw p < 0.05: {(g63['p_raw'] < 0.05).sum()} · bootstrap CI excludes 0: "
      f"{g63['boot_ci_excludes_zero'].sum()} · Holm-adjusted p < 0.05: {(g63['p_holm'] < 0.05).sum()} "
      f"(about {0.05 * len(g63):.0f} expected by chance at 5%)")
display(g63.sort_values("p_raw").head(12)[["window_months", "perception_measure", "fundamental", "r", "ci_low", "ci_high",
                                           "n_year_pairs", "p_raw", "p_holm"]].style.format(precision=3).hide(axis="index"))

### 13d. Do the three surveys agree? *(robustness: "same direction in all three surveys")*
One row per change of party, the winning-minus-losing gap change in each source over its own natural window. Levels and
magnitudes are **not** comparable across sources (different questions and units), so the test is direction and rough
order of magnitude only.

In [0]:
conc = []
for ev in ["pres-2016", "pres-2020", "pres-2024"]:
    row = {"transition": NAMES[ev].replace(" presidential", "")}
    ce = contrasts[(contrasts["source"] == "CES") & (contrasts["event_key"] == ev) & (contrasts["measure"] == "pct_better")
                   & (contrasts["party_definition"] == "Leaners with parties") & (contrasts["comparison_type"] == "Presidential transition")]
    row["CES (next fall wave, pp)"] = ce["winner_minus_loser_gap_change"].iloc[0] if len(ce) else np.nan
    for win, lab in [(PEW_BRACKET, "Pew (last before → first after, pp)"), (PEW_6M, "Pew (pre-election → 6 months, pp)")]:
        p = contrasts[(contrasts["source"] == "Pew") & (contrasts["event_key"] == ev) & (contrasts["comparison_window"] == win)]
        row[lab] = p["winner_minus_loser_gap_change"].iloc[0] if len(p) else np.nan
    u = placebo[(placebo["source"] == "UMich (monthly)") & (placebo["event_key"] == ev) & (placebo["checkpoint_months_from_election"] == 8)]
    row["UMich (baseline → +8 months, index pts)"] = u["winner_minus_loser_gap_change"].iloc[0] if len(u) else np.nan
    vals = [v for k, v in row.items() if k != "transition" and pd.notna(v)]
    row["all sources same direction"] = all(v > 0 for v in vals)
    conc.append(row)
concordance = pd.DataFrame(conc)
display(concordance.style.format(precision=1, na_rep="— (not covered)").hide(axis="index"))
print("Note: 2016 Pew windows rest on one poll dated Election Day; UMich party series starts April 2017.")
if SAVE_OUTPUTS:
    concordance.to_csv(TABLE_DIR / "cross_source_concordance.csv", index=False)

### 13e. The gap is the widest it has been *(descriptive)*
Yearly average of the Republican-minus-Democrat UMich gap. Positive = Republicans more upbeat. The sign tracks the White
House; the **size** is the news: 2025–26 exceed every earlier year in either direction. Bars carry the color of the party
holding the White House that year (the year's majority of months), not a party's "score".

In [0]:
gy = (monthly[(monthly["series"] == "Republican minus Democrat") & monthly["sentiment_index"].notna()]
      .groupby("year").agg(gap=("sentiment_index", "mean"), months=("sentiment_index", "size"),
                           wh=("white_house_party", lambda s: s.mode().iloc[0])).reset_index())
fig, ax = plt.subplots(figsize=(10.5, 4.4))
ax.bar(gy["year"], gy["gap"], color=[COLORS[p] for p in gy["wh"]], width=0.62)
for _, r in gy.iterrows():
    ax.text(r["year"], r["gap"] + (1.5 if r["gap"] >= 0 else -1.5), f"{r['gap']:+.0f}" + ("*" if r["months"] < 12 else ""),
            ha="center", va="bottom" if r["gap"] >= 0 else "top", fontsize=8.5, color=INK)
ax.axhline(0, color=MUTED, lw=0.8)
ax.set_ylim(gy["gap"].min() - 10, gy["gap"].max() + 10)    # room for the value labels at both ends
ax.set_xticks(gy["year"])
ax.set_ylabel("Rep − Dem sentiment gap (index pts)")
ax.legend(handles=[Patch(color=COLORS["Republican"], label="Republican White House"),
                   Patch(color=COLORS["Democrat"], label="Democratic White House")], loc="lower left", fontsize=8.5)
add_titles(fig, "The partisan gap in economic sentiment is the widest in the series",
           "Average monthly gap between Republicans' and Democrats' UMich sentiment index, by year. Positive = Republicans more upbeat.")
add_source(fig, f"Source: {SOURCE_UMICH}. *Partial year (2017 starts in April; current year to latest month).")
layout(fig, top=0.86, bottom=0.08)
save(fig, "17_partisan_gap_by_year")

## 14. Story numbers → draft nut graf
The flagship's opening paragraphs, with **every number pulled from the number sheet** (`N[...]`), so the prose and the
tables cannot disagree. This is a scaffold for the first 400 words, not finished copy: the wording is for the writer;
the numbers are for the pipeline. Rounding follows the newsroom convention (whole numbers for shares in prose, one
decimal only where the decimal carries meaning).

In [0]:
draft = f"""
**The economy didn't change when the White House did. Half the country's opinion of it did.**

In the fall of 2024, about {N['ces_rep_2024']:.0f}% of Republicans told the Cooperative Election Study that the
national economy had gotten better over the past year. A year later, with their party back in the White House,
{N['ces_rep_2025']:.0f}% said so. Democrats made the opposite trip, from {N['ces_dem_2024']:.0f}% to {N['ces_dem_2025']:.0f}%.
Independents barely moved ({N['ces_ind_2024']:.0f}% to {N['ces_ind_2025']:.0f}%).

The economy itself did not make that trip. Inflation was {N['infl_sep24']:.1f}% in September 2024 and {N['infl_jan25']:.1f}%
in the month of the inauguration. Over the same months, each party's consumer sentiment moved by
{N['sent_move_min']:.0f} to {N['sent_move_max']:.0f} points on the University of Michigan's index.

Both parties do this, and about equally. Holding the year (and so the economy) fixed, having your party in the
White House makes a person about {N['rq1_premium']:.0f} percentage points more likely to say the economy is improving:
{N['rq1_rep']:.0f} points for Republicans, {N['rq1_dem']:.0f} for Democrats. Change the survey weights, the coding of
"not sure", the treatment of independents who lean to a party, or drop the pandemic years, and the number stays between
{N['rq1_min']:.0f} and {N['rq1_max']:.0f}.

The swing does not wait for the new president. In 2024–25, about {N['sw_rep24_e']:.0f}% of the Republican shift had
happened by the election month and {N['sw_rep24_t']:.0f}% by the inauguration; the rest came over the following six months.

What people say about their own finances is a different story. In the American National Election Studies, the partisan
gap on "the economy" was {N['rq3_ratio24']:.1f} times the gap on respondents' own finances in 2024, and
{N['rq3_ratio20']:.1f} times in 2020. Your wallet, it turns out, is far less partisan than your view of the country.
"""
display(Markdown(draft))
if SAVE_OUTPUTS:
    (TABLE_DIR / "draft_nut_graf.md").write_text(draft.strip() + "\n")

### 14a. Caveats the story must carry (from the handoff, said out loud)
* **Two transitions** in the monthly data: "in both 2020–21 and 2024–25", never "on average".
* **Democrats lagged in 2024–25** (0% of their swing done at the election): one transition, so a question, not a claim.
* **2017** combined a common improvement (independents +26) with the partisan swing.
* **CES annual midterm comparisons** sit inside the quiet-year drift; UMich monthly is the placebo that carries weight.
* **Pew 2024–25 bracket window understates the flip** (no poll between election and inauguration): the pre-election →
  six-months window is the headline, both shown.
* **Expressive responding** ("cheerleading") cannot be separated from sincere belief with these data; paid-accuracy
  experiments shrink the gap but do not erase it.
* **Open data gaps:** ANES strata/PSU (design-based SEs), Pew sample sizes, UMich subgroup n and expectations by party,
  official UNRATE, 2015 CPI levels, county unemployment, recent-layoff control in `int_respondents`, party-composition
  reweighting.

## 15. Exports and session info
Charts were saved as each was drawn (PNG for slides, SVG for the web and for hand edits). This cell writes the number
sheet and, optionally, **parquet copies of every mart** so the repo can redraw every chart without Databricks
(set `EXPORT_MARTS = True` once, commit `marts/`, and run this notebook locally with `LOCAL_MARTS_DIR=marts`).

In [0]:
EXPORT_MARTS = False

if SAVE_OUTPUTS:
    number_sheet.to_csv(TABLE_DIR / "number_sheet.csv", index=False)
    if EXPORT_MARTS:
        mart_dir = OUTPUT_DIR / "marts"
        mart_dir.mkdir(parents=True, exist_ok=True)
        for name, df in M.items():
            df.to_parquet(mart_dir / f"{name}.parquet", index=False)
    files = sorted(p.relative_to(OUTPUT_DIR).as_posix() for p in OUTPUT_DIR.rglob("*") if p.is_file())
    print(f"{len(files)} files in {OUTPUT_DIR}:")
    print("\n".join("  " + f for f in files))

session = {
    "run_at_utc": datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M"),
    "environment": "Databricks" if IN_DATABRICKS else "local",
    "pandas": pd.__version__, "numpy": np.__version__, "matplotlib": matplotlib.__version__,
    "numbers_matching_handoff": f"{int(number_sheet['matches_handoff'].sum())}/{len(number_sheet)}",
    "latest_umich_month": f"{monthly['month'].max():%Y-%m}",
}
display(pd.Series(session, name="session").to_frame())